# Read, not reasoned? Kaggle run-all

**Setup:** *Settings -> Accelerator -> GPU T4 x2*, *Settings -> Internet -> On*, then **Save Version -> Run All**.
Two notebooks can run at the same time (each uses its own 2 GPUs):

* **A_reliance_small** (~8 h): estimator benchmark for Qwen2.5-0.5B/1.5B/3B and SmolLM2-1.7B with the
  round-3 additions: resampled-baseline rollout, wrong-answer copy test (conditioned on chains the model
  solves), volume-matched mask control, raw log-prob effects.
* **B_7b_and_mediation** (~8 h): Qwen2.5-7B estimator benchmark (sharded over both GPUs), then the exact
  mediation analysis on MuSiQue at n=500 for Qwen2.5-1.5B and 3B.

No job starts after `MAX_START_HOURS`, so the session never hits Kaggle's 12-hour limit (which would
discard all output). Download `results_<notebook>.zip` at the end.


In [ ]:
!pip -q uninstall -y torchao 2>/dev/null | tail -1
!pip -q install -U "transformers>=4.44" accelerate scikit-learn peft 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv || true

In [ ]:
import os
os.makedirs('hgcd', exist_ok=True)
os.environ.setdefault('HGCD_DATA', os.path.abspath('hgcd_data'))

In [ ]:
%%writefile hgcd/__init__.py
"""HGCD: Hop-Graph Contrastive Decoding for multi-hop reasoning.

Training-free, inference-time interventions on a model's own reasoning graph.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile hgcd/data.py
"""Dataset loading and controlled (binary) interventions.

CLUTRR  : kinship chains with a gold path graph (controlled domain).
MuSiQue : real multi-hop QA with gold decompositions (bridge-entity chains).

Every intervention is a deterministic, seeded program (no LLM involved), so the
ground-truth effect of each intervention is known by construction:
  rename / order / distract / placebo_input  ->  true effect on the answer = 0.
"""

from __future__ import annotations

import ast
import json
import os
import random
import re
import urllib.request
from dataclasses import dataclass, field, replace
from typing import Dict, List, Optional, Tuple

CLUTRR_URL = ("https://raw.githubusercontent.com/kliang5/CLUTRR_huggingface_dataset/"
              "main/{task}/{split}.csv")
CLUTRR_TASK = "gen_train234_test2to10"

# Fictional names (not used by CLUTRR) for renaming / distractor families.
FRESH_NAMES = {
    "male": ["Orrin", "Thadeus", "Brannock", "Celwyn", "Doran", "Eldric", "Fennick", "Garrow",
             "Halvard", "Ivor", "Jorvik", "Kestrel", "Lorcan", "Madoc", "Norris", "Osric",
             "Perrin", "Quillon", "Roderick", "Soren", "Tobias", "Ulric", "Varek", "Wystan",
             "Yorick", "Zander", "Alaric", "Bastian", "Corwin", "Dashiel"],
    "female": ["Aveline", "Brisa", "Calla", "Delphine", "Elowen", "Fenna", "Giselle", "Hesper",
               "Isolde", "Junia", "Kerensa", "Liora", "Maelis", "Nerys", "Oriel", "Perpetua",
               "Rosalind", "Sabine", "Talitha", "Undine", "Verity", "Winsome", "Xanthe", "Yseult",
               "Zinnia", "Amabel", "Briallen", "Cressida", "Damaris", "Evadne"],
}

NEUTRAL_SENTENCES = [
    "It was a sunny afternoon.",
    "The town had a small library near the river.",
    "Everyone agreed that the soup was delicious.",
    "The bus arrived a few minutes late that day.",
    "A light rain fell during the evening.",
]


# ----------------------------------------------------------------------------- utils

def _cache_dir() -> str:
    d = os.environ.get("HGCD_DATA", os.path.join(os.path.expanduser("~"), ".cache", "hgcd"))
    os.makedirs(d, exist_ok=True)
    return d


def _download(url: str, path: str) -> str:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path


def strip_brackets(text: str) -> str:
    return text.replace("[", "").replace("]", "")


def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if p]


def _sub_names(text: str, mapping: Dict[str, str]) -> str:
    """Replace bracketed CLUTRR names ([Name]) according to mapping."""
    def rep(m):
        n = m.group(1)
        return "[" + mapping.get(n, n) + "]"
    return re.sub(r"\[([^\]]+)\]", rep, text)


# ----------------------------------------------------------------------------- CLUTRR

@dataclass
class GoldHop:
    head: str
    rel: str
    tail: str

    def text(self) -> str:
        return f"{self.head}'s {self.rel} is {self.tail}."


@dataclass
class ClutrrExample:
    uid: str
    story: str                    # raw story with [Name] brackets
    names: List[str]              # entity names, index order
    genders: Dict[str, str]
    source: str                   # query[0]
    target: str                   # query[1]
    answer: str                   # relation: target is source's <answer>
    gold_hops: List[GoldHop]
    condition: str = "orig"
    extra_facts: List[GoldHop] = field(default_factory=list)  # true facts off the path

    @property
    def k(self) -> int:
        return len(self.gold_hops)

    @property
    def story_text(self) -> str:
        return strip_brackets(self.story)

    def question(self) -> str:
        return f"How is {self.target} related to {self.source}?"

    def answer_prefix(self) -> str:
        return f"Answer: {self.target} is {self.source}'s"


def _parse_clutrr_row(row: dict) -> Optional[ClutrrExample]:
    edges = ast.literal_eval(row["story_edges"])
    qedge = ast.literal_eval(row["query_edge"])
    if edges != [(i, i + 1) for i in range(len(edges))] or qedge != (0, len(edges)):
        return None  # keep only clean simple-path stories (documented filter)
    rels = ast.literal_eval(row["edge_types"])
    gpairs = [g.split(":") for g in row["genders"].split(",")]
    names = [g[0] for g in gpairs]
    genders = {g[0]: g[1] for g in gpairs}
    if len(names) != len(edges) + 1:
        return None
    hops = [GoldHop(names[i], rels[i], names[i + 1]) for i in range(len(edges))]
    return ClutrrExample(uid=row["id"], story=row["story"], names=names, genders=genders,
                         source=names[0], target=names[-1], answer=row["target_text"],
                         gold_hops=hops)


def load_clutrr(split: str = "test", task: str = CLUTRR_TASK) -> List[ClutrrExample]:
    import csv
    path = os.path.join(_cache_dir(), f"clutrr_{task}_{split}.csv")
    _download(CLUTRR_URL.format(task=task, split=split), path)
    out = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ex = _parse_clutrr_row(row)
            if ex is not None:
                out.append(ex)
    return out


def clutrr_relations(examples: List[ClutrrExample]) -> List[str]:
    rels = set()
    for ex in examples:
        rels.add(ex.answer)
        rels.update(h.rel for h in ex.gold_hops)
    return sorted(rels)


def sample_by_k(examples: List[ClutrrExample], ks: List[int], n_per_k: int,
                seed: int = 0) -> List[ClutrrExample]:
    rng = random.Random(seed)
    out = []
    for k in ks:
        pool = [e for e in examples if e.k == k]
        rng.shuffle(pool)
        out.extend(pool[:n_per_k])
    return out


# --- interventions (all binary, seeded, ground-truth effect on the answer = 0)

def _fresh_mapping(names: List[str], genders: Dict[str, str], rng: random.Random,
                   avoid: Optional[set] = None) -> Dict[str, str]:
    avoid = set(avoid or ())
    pools = {g: [n for n in FRESH_NAMES[g] if n not in avoid] for g in FRESH_NAMES}
    for g in pools:
        rng.shuffle(pools[g])
    mapping = {}
    for n in names:
        g = genders.get(n, "male")
        mapping[n] = pools[g].pop() if pools[g] else n + "x"
    return mapping


def rename(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"rename-{ex.uid}-{seed}")
    m = _fresh_mapping(ex.names, ex.genders, rng)
    return replace(ex, story=_sub_names(ex.story, m), names=[m[n] for n in ex.names],
                   genders={m[n]: g for n, g in ex.genders.items()},
                   source=m[ex.source], target=m[ex.target],
                   gold_hops=[GoldHop(m[h.head], h.rel, m[h.tail]) for h in ex.gold_hops],
                   condition="rename")


def reorder(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"order-{ex.uid}-{seed}")
    sents = split_sentences(ex.story)
    if len(sents) > 1:
        orig = list(sents)
        for _ in range(10):
            rng.shuffle(sents)
            if sents != orig:
                break
    return replace(ex, story=" ".join(sents), condition="order")


def _distractor_family(ex: ClutrrExample, pool: List[ClutrrExample], rng: random.Random):
    """A disconnected family from another story, renamed with fresh names."""
    other = rng.choice([o for o in pool if o.uid != ex.uid and o.k <= 3] or pool)
    m = _fresh_mapping(other.names, other.genders, rng, avoid=set(ex.names))
    story = _sub_names(other.story, m)
    facts = [GoldHop(m[h.head], h.rel, m[h.tail]) for h in other.gold_hops]
    return story, facts, {m[n]: g for n, g in other.genders.items()}


def distract(ex: ClutrrExample, pool: List[ClutrrExample], seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"distract-{ex.uid}-{seed}")
    dstory, facts, dgen = _distractor_family(ex, pool, rng)
    sents = split_sentences(ex.story)
    for s in split_sentences(dstory):
        sents.insert(rng.randint(0, len(sents)), s)
    g = dict(ex.genders)
    g.update(dgen)
    return replace(ex, story=" ".join(sents), genders=g, extra_facts=facts, condition="distract")


def placebo_input(ex: ClutrrExample, seed: int = 0) -> ClutrrExample:
    rng = random.Random(f"placebo-{ex.uid}-{seed}")
    sents = split_sentences(ex.story)
    for s in rng.sample(NEUTRAL_SENTENCES, 2):
        sents.insert(rng.randint(0, len(sents)), s)
    return replace(ex, story=" ".join(sents), condition="placebo_input")


def counterfactual_story(ex: ClutrrExample, seed: int = 0) -> str:
    """Type-matched counterfactual context for L1 contrast.

    Same sentences, same length, same entity *set* and genders, but names are permuted
    within each gender group (cyclic shift), so who-is-related-to-whom changes. Singleton
    gender groups are mapped to a fresh name of the same gender.
    """
    rng = random.Random(f"cf-{ex.uid}-{seed}")
    groups: Dict[str, List[str]] = {}
    for n in ex.names:
        groups.setdefault(ex.genders.get(n, "male"), []).append(n)
    m: Dict[str, str] = {}
    for g, ns in groups.items():
        if len(ns) >= 2:
            sh = list(ns)
            rng.shuffle(sh)
            for a, b in zip(sh, sh[1:] + sh[:1]):
                m[a] = b
        else:
            m[ns[0]] = rng.choice([x for x in FRESH_NAMES[g] if x not in ex.names])
    return strip_brackets(_sub_names(ex.story, m))


def apply_condition(ex: ClutrrExample, cond: str, pool: List[ClutrrExample],
                    seed: int = 0) -> ClutrrExample:
    if cond == "orig":
        return ex
    if cond == "rename":
        return rename(ex, seed)
    if cond == "order":
        return reorder(ex, seed)
    if cond == "distract":
        return distract(ex, pool, seed)
    if cond == "placebo_input":
        return placebo_input(ex, seed)
    raise ValueError(cond)


def with_redundant_fact(ex: ClutrrExample, pool: List[ClutrrExample],
                        seed: int = 0) -> Tuple[ClutrrExample, GoldHop]:
    """Add a disconnected family to the story and return one true-but-irrelevant fact.

    Used to build gold chains with an inserted redundant hop whose true necessity is 0.
    """
    ex2 = distract(ex, pool, seed)
    return ex2, ex2.extra_facts[0]


# ----------------------------------------------------------------------------- MuSiQue

@dataclass
class MusiqueExample:
    uid: str
    question: str
    answer: str
    aliases: List[str]
    paragraphs: List[dict]        # {'title','paragraph_text','is_supporting'}
    gold_hops: List[Tuple[str, str]]   # (sub-question rendered, sub-answer)
    condition: str = "orig"
    n_dist: int = 4
    order_seed: int = 0

    @property
    def k(self) -> int:
        return len(self.gold_hops)

    def context(self) -> List[dict]:
        sup = [p for p in self.paragraphs if p["is_supporting"]]
        dis = [p for p in self.paragraphs if not p["is_supporting"]]
        rng = random.Random(f"ms-dist-{self.uid}")
        rng.shuffle(dis)
        paras = sup + dis[: self.n_dist]
        rng2 = random.Random(f"ms-order-{self.uid}-{self.order_seed}")
        rng2.shuffle(paras)
        return paras

    def context_text(self) -> str:
        return "\n".join(f"[{i + 1}] {p['title']}: {p['paragraph_text']}"
                         for i, p in enumerate(self.context()))

    def distractor_titles(self) -> List[str]:
        return [p["title"] for p in self.context() if not p["is_supporting"]]

    def answer_prefix(self) -> str:
        return "Answer:"


def _render_subq(q: str, answers: List[str]) -> str:
    for i, a in enumerate(answers):
        q = q.replace(f"#{i + 1}", a)
    if ">>" in q:
        ent, rel = [s.strip() for s in q.split(">>", 1)]
        return f"What is the {rel} of {ent}?"
    q = q.strip()
    return q if q.endswith("?") else q + "?"


def load_musique(split: str = "dev") -> List[MusiqueExample]:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("bdsaglam/musique", f"musique_ans_v1.0_{split}.jsonl", repo_type="dataset")
    out = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            answers = [s["answer"] for s in d["question_decomposition"]]
            hops = [(_render_subq(s["question"], answers[:i]), s["answer"])
                    for i, s in enumerate(d["question_decomposition"])]
            out.append(MusiqueExample(uid=d["id"], question=d["question"], answer=d["answer"],
                                      aliases=d.get("answer_aliases") or [],
                                      paragraphs=d["paragraphs"], gold_hops=hops))
    return out


def musique_condition(ex: MusiqueExample, cond: str) -> MusiqueExample:
    """MuSiQue interventions: number of distractor paragraphs and paragraph order."""
    if cond == "orig":
        return replace(ex, condition="orig", n_dist=4, order_seed=0)
    if cond == "dist0":
        return replace(ex, condition="dist0", n_dist=0)
    if cond == "dist8":
        return replace(ex, condition="dist8", n_dist=8)
    if cond == "order":
        return replace(ex, condition="order", order_seed=1)
    raise ValueError(cond)


def map_names(text: str, src: ClutrrExample, dst: ClutrrExample) -> str:
    """Rewrite a chain produced for `src` into `dst`'s entity names (identity unless renamed).
    Entities are aligned by index, which `rename` preserves."""
    if src.names == dst.names[:len(src.names)]:
        return text
    m = {a: b for a, b in zip(src.names, dst.names)}
    return re.sub(r"\b(" + "|".join(map(re.escape, sorted(m, key=len, reverse=True))) + r")\b",
                  lambda x: m[x.group(1)], text)


In [ ]:
%%writefile hgcd/chain.py
"""Hop-structured chains, the Directed Reasoning Graph (DRG) and do()-interventions.

A chain is a list of hops rendered as lines

    Hop i: <body> => <result>

DRG edge i -> j  (i < j)  iff  result_i is mentioned in body_j.

Interventions on hop j
  do_naive(j)  : mask hop j only. Estimates the *controlled direct effect* of h_j on the
                 answer, holding all downstream hops fixed.
  do_total(j)  : mask hop j and, in every DRG-descendant d of j, mask every mention of
                 result_j and of result_d (descendant values are functions of h_j, so under
                 do(h_j) they are unknown). Deterministic approximation of the *total effect*
                 without re-generating the downstream chain.
  placebo(j)   : content-preserving rewrite of hop j (true effect = 0).
"""

from __future__ import annotations

import re
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Set

MASK = "[MASKED]"
HOP_RE = re.compile(r"^\s*Hop\s*(\d+)\s*:\s*(.*?)\s*=>\s*(.+?)\s*$")


@dataclass
class Hop:
    idx: int
    body: str
    result: str


def _clean_result(r: str) -> str:
    r = r.strip().strip(".").strip()
    r = re.sub(r"\s*\(.*?\)\s*$", "", r)  # drop trailing "(relation)" annotations
    return r.strip()


def parse_chain(text: str) -> List[Hop]:
    hops = []
    for line in text.splitlines():
        if line.strip().lower().startswith("answer"):
            break
        m = HOP_RE.match(line)
        if m:
            res = _clean_result(m.group(3))
            if res:
                hops.append(Hop(int(m.group(1)), m.group(2).strip(), res))
    return hops


def render_chain(hops: List[Hop]) -> str:
    return "".join(f"Hop {i + 1}: {h.body} => {h.result}\n" for i, h in enumerate(hops))


def _mentions(text: str, ent: str) -> bool:
    if not ent or ent == MASK or len(ent) < 2:
        return False
    return re.search(_ent_pattern(ent), text, flags=re.IGNORECASE) is not None


def _ent_pattern(ent: str) -> str:
    pre = r"\b" if ent[0].isalnum() else ""
    post = r"\b" if ent[-1].isalnum() else ""
    return pre + re.escape(ent) + post


def _mask_ents(text: str, ents: Set[str]) -> str:
    for e in sorted(ents, key=len, reverse=True):
        if e and e != MASK and len(e) >= 2:
            text = re.sub(_ent_pattern(e), MASK, text, flags=re.IGNORECASE)
    return text


# ----------------------------------------------------------------------------- DRG

def drg_edges(hops: List[Hop]) -> List[tuple]:
    return [(i, j) for j in range(len(hops)) for i in range(j)
            if _mentions(hops[j].body, hops[i].result)]


def descendants(hops: List[Hop], j: int) -> List[int]:
    children: Dict[int, List[int]] = {}
    for a, b in drg_edges(hops):
        children.setdefault(a, []).append(b)
    seen, stack = set(), [j]
    while stack:
        for c in children.get(stack.pop(), []):
            if c not in seen:
                seen.add(c)
                stack.append(c)
    return sorted(seen)


def ancestors(hops: List[Hop], j: int) -> List[int]:
    parents: Dict[int, List[int]] = {}
    for a, b in drg_edges(hops):
        parents.setdefault(b, []).append(a)
    seen, stack = set(), [j]
    while stack:
        for p in parents.get(stack.pop(), []):
            if p not in seen:
                seen.add(p)
                stack.append(p)
    return sorted(seen)


def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()


def relevant_hops(hops: List[Hop], target: Optional[str] = None) -> List[int]:
    """Hops with a directed DRG path to the answer (ancestors-or-self of terminal hops).

    Terminal hops: hops whose result is the query target (CLUTRR); else the last hop.
    """
    if not hops:
        return []
    terms = [i for i, h in enumerate(hops) if target and norm(h.result) == norm(target)]
    if not terms:
        terms = [len(hops) - 1]
    rel = set()
    for t in terms:
        rel.add(t)
        rel.update(ancestors(hops, t))
    return sorted(rel)


# ----------------------------------------------------------------------------- do()

def do_naive(hops: List[Hop], j: int) -> List[Hop]:
    out = list(hops)
    out[j] = replace(out[j], body=MASK, result=MASK)
    return out


def do_total(hops: List[Hop], j: int) -> List[Hop]:
    desc = descendants(hops, j)
    ents = {hops[j].result} | {hops[d].result for d in desc}
    out = list(hops)
    out[j] = replace(out[j], body=MASK, result=MASK)
    for d in desc:
        out[d] = replace(out[d], body=_mask_ents(out[d].body, ents),
                         result=_mask_ents(out[d].result, ents))
    return out


CLUTRR_HOP_RE = re.compile(r"^(.+?)'s ([\w-]+) is (.+?)\.?$")


def placebo(hops: List[Hop], j: int) -> List[Hop]:
    """Content-preserving rewrite of hop j (same entities, same relation)."""
    out = list(hops)
    b = hops[j].body
    m = CLUTRR_HOP_RE.match(b)
    if m:
        nb = f"{m.group(3)} is the {m.group(2)} of {m.group(1)}."
    elif b.endswith("?"):
        nb = "To restate: " + b
    else:
        nb = "In other words, " + b
    out[j] = replace(out[j], body=nb)
    return out


def mask_marker(hops: List[Hop], j: int) -> List[Hop]:
    """Artifact control: insert the mask token into hop j WITHOUT removing any content.
    Its effect isolates the out-of-distribution cost of the [MASKED] token itself."""
    out = list(hops)
    out[j] = replace(out[j], body=f"{MASK} {hops[j].body}")
    return out


def replace_result(hops: List[Hop], j: int, new: str, propagate: bool) -> List[Hop]:
    """Counterfactual replacement (in-distribution, no mask token on hop j): hop j yields `new`.

    propagate=False keeps downstream hops as written (controlled direct effect).
    propagate=True rewrites mentions of the old value in every DRG descendant AND invalidates the
    descendants' own results, which are functions of h_j and therefore unknown under the
    intervention. Renaming alone is not enough: on bridge chains the last hop still states the
    answer, so a rename-only "total" estimator collapses to the direct effect."""
    old = hops[j].result
    out = list(hops)
    out[j] = replace(out[j], body=_sub_ent(out[j].body, old, new), result=new)
    if propagate:
        desc = descendants(hops, j)
        stale = {hops[d].result for d in desc}
        for d in desc:
            body = _mask_ents(_sub_ent(out[d].body, old, new), stale)
            out[d] = replace(out[d], body=body, result=MASK)
    return out


def _sub_ent(text: str, old: str, new: str) -> str:
    if not old or len(old) < 2:
        return text
    return re.sub(_ent_pattern(old), new, text, flags=re.IGNORECASE)


def mask_marker_tree(hops: List[Hop], j: int) -> List[Hop]:
    """Volume-matched artifact control for do_total: insert [MASKED] into hop j AND every DRG
    descendant while keeping all content. Same number of mask tokens in the same places as the
    total-effect intervention, but no information removed."""
    out = list(hops)
    for d in [j] + descendants(hops, j):
        out[d] = replace(out[d], body=f"{MASK} {hops[d].body}", result=f"{hops[d].result} {MASK}")
    return out


In [ ]:
%%writefile hgcd/prompts.py
"""Few-shot prompts that elicit hop-structured chains."""

from __future__ import annotations

from typing import List

from .chain import Hop, render_chain

CLUTRR_SYSTEM = (
    "You solve family-relation puzzles. Walk from the first person to the second person "
    "one relation at a time, using only facts stated in the story. Write one line per hop "
    "in the form 'Hop i: X's <relation> is Y. => Y', then give the final answer.")

CLUTRR_SHOTS = [
    {
        "story": "Mabel took her son Rupert to the zoo. Rupert and his sister Agatha baked a cake "
                 "for their grandmother.",
        "src": "Mabel", "tgt": "Agatha",
        "hops": [("Mabel's son is Rupert.", "Rupert"), ("Rupert's sister is Agatha.", "Agatha")],
        "ans": "daughter",
    },
    {
        "story": "Clement went fishing with his father Bertram. Bertram called his sister Winifred "
                 "to wish her a happy birthday. Winifred was proud of her daughter Odette.",
        "src": "Clement", "tgt": "Odette",
        "hops": [("Clement's father is Bertram.", "Bertram"),
                 ("Bertram's sister is Winifred.", "Winifred"),
                 ("Winifred's daughter is Odette.", "Odette")],
        "ans": "cousin",
    },
]
# The second shot uses a relation outside the CLUTRR label set on purpose: it teaches the
# format without leaking label statistics. Its answer is never a scoring candidate.

MUSIQUE_SYSTEM = (
    "You answer multi-hop questions using the given paragraphs. Break the question into "
    "single-hop sub-questions, answer each in one line of the form "
    "'Hop i: <sub-question> => <answer>', then give the final answer.")

MUSIQUE_SHOTS = [
    {
        "context": "[1] Thornfield Abbey: Thornfield Abbey is a novel written by Edwina Marsh.\n"
                   "[2] Edwina Marsh: Edwina Marsh was born in Galway in 1902.\n"
                   "[3] Galway Bay: Galway Bay is a large bay on the west coast of Ireland.",
        "question": "Where was the author of Thornfield Abbey born?",
        "hops": [("Who is the author of Thornfield Abbey?", "Edwina Marsh"),
                 ("Where was Edwina Marsh born?", "Galway")],
        "ans": "Galway",
    },
]


def _clutrr_user(story: str, src: str, tgt: str) -> str:
    return f"Story: {story}\nQuestion: How is {tgt} related to {src}?"


def _hops_text(pairs) -> str:
    return render_chain([Hop(i + 1, b, r) for i, (b, r) in enumerate(pairs)])


def clutrr_messages(story: str, src: str, tgt: str) -> List[dict]:
    msgs = [{"role": "system", "content": CLUTRR_SYSTEM}]
    for s in CLUTRR_SHOTS:
        msgs.append({"role": "user", "content": _clutrr_user(s["story"], s["src"], s["tgt"])})
        msgs.append({"role": "assistant", "content": _hops_text(s["hops"]) +
                     f"Answer: {s['tgt']} is {s['src']}'s {s['ans']}."})
    msgs.append({"role": "user", "content": _clutrr_user(story, src, tgt)})
    return msgs


def musique_messages(context: str, question: str) -> List[dict]:
    msgs = [{"role": "system", "content": MUSIQUE_SYSTEM}]
    for s in MUSIQUE_SHOTS:
        msgs.append({"role": "user", "content": f"Paragraphs:\n{s['context']}\nQuestion: {s['question']}"})
        msgs.append({"role": "assistant", "content": _hops_text(s["hops"]) + f"Answer: {s['ans']}"})
    msgs.append({"role": "user", "content": f"Paragraphs:\n{context}\nQuestion: {question}"})
    return msgs


def musique_question_only(question: str) -> List[dict]:
    """No-context prompt (for CAD's contrast distribution)."""
    return musique_messages("(no paragraphs given)", question)


def clutrr_question_only(src: str, tgt: str) -> List[dict]:
    return clutrr_messages("(no story given)", src, tgt)


In [ ]:
%%writefile hgcd/lm.py
"""Thin HF-transformers wrapper: chat prompts, batched generation, KV-cached
candidate scoring, and contrastive (two-context) greedy decoding."""

from __future__ import annotations

import math
from typing import List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F


class LM:
    def __init__(self, name: str, device: Optional[str] = None, dtype=None,
                 device_map: Optional[str] = None):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.name = name
        self.tok = AutoTokenizer.from_pretrained(name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        cuda = torch.cuda.is_available()
        dtype = dtype or (torch.float16 if cuda else torch.float32)
        import transformers
        kw = dict(device_map=device_map) if device_map else {}
        major_minor = tuple(int(x) for x in transformers.__version__.split(".")[:2])
        kw["dtype" if major_minor >= (4, 56) else "torch_dtype"] = dtype
        self.model = AutoModelForCausalLM.from_pretrained(name, **kw)
        if next(self.model.parameters()).dtype != dtype:  # never silently fall back to fp32
            self.model.to(dtype)
        if not device_map:
            self.model.to(device or ("cuda" if cuda else "cpu"))
        self.model.eval()
        self.device = next(self.model.parameters()).device
        self.n_boundary_fallback = 0
        cfg = self.model.config
        heads = getattr(cfg, "num_attention_heads", 16)
        kvh = getattr(cfg, "num_key_value_heads", None) or heads
        hd = getattr(cfg, "head_dim", None) or cfg.hidden_size // heads
        bytes_el = torch.finfo(dtype).bits // 8
        # KV-cache bytes per token: models without GQA (e.g. SmolLM2) need ~10x more than Qwen2.5
        self.kv_bytes_per_token = 2 * cfg.num_hidden_layers * kvh * hd * bytes_el
        self.vocab = getattr(cfg, "vocab_size", 150000)
        import os
        self.mem_budget = float(os.environ.get("HGCD_MEM_BUDGET_GB", "1.5")) * 2 ** 30

    def _safe_chunk(self, chunk: int, prefix_len: int, cand_len: int) -> int:
        kv = self.kv_bytes_per_token * (prefix_len + cand_len)
        logits = 4 * cand_len * self.vocab
        return max(1, min(chunk, int(self.mem_budget // max(1, kv + logits))))

    # ------------------------------------------------------------------ prompts
    def prompt(self, messages: List[dict], assistant_prefix: str = "", thinking: bool = False) -> str:
        kw = {}
        if "enable_thinking" in (getattr(self.tok, "chat_template", None) or ""):   # Qwen3 family
            kw["enable_thinking"] = bool(thinking)
        p = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
        return p + assistant_prefix

    def _enc(self, texts: Sequence[str]):
        return self.tok(list(texts), return_tensors="pt", padding=True,
                        add_special_tokens=False).to(self.device)

    # ------------------------------------------------------------------ generation
    @torch.no_grad()
    def generate(self, prompts: List[str], max_new_tokens: int = 160, do_sample: bool = False,
                 temperature: float = 0.7, n: int = 1, batch_size: int = 8,
                 stop: str = "Answer:") -> List[str]:
        try:
            return self._generate(prompts, max_new_tokens, do_sample, temperature, n, batch_size, stop)
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            torch.cuda.empty_cache()
            return self.generate(prompts, max_new_tokens, do_sample, temperature, n,
                                 max(1, batch_size // 2), stop)

    def _generate(self, prompts, max_new_tokens, do_sample, temperature, n, batch_size, stop):
        outs = []
        for i in range(0, len(prompts), batch_size):
            enc = self._enc(prompts[i:i + batch_size])
            kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                      num_return_sequences=n, pad_token_id=self.tok.pad_token_id)
            if do_sample:
                kw.update(temperature=temperature, top_p=1.0, top_k=0)
            else:
                kw.update(temperature=None, top_p=None, top_k=None)
            try:
                g = self.model.generate(**enc, stop_strings=[stop], tokenizer=self.tok, **kw)
            except (TypeError, ValueError):
                g = self.model.generate(**enc, **kw)
            new = g[:, enc.input_ids.shape[1]:]
            for t in self.tok.batch_decode(new, skip_special_tokens=True):
                outs.append(t.split(stop)[0])
        return outs

    # ------------------------------------------------------------------ scoring
    def _cand_ids(self, p_ids: List[int], prefix: str, cands: Sequence[str]) -> List[List[int]]:
        out = []
        for c in cands:
            full = self.tok(prefix + c, add_special_tokens=False).input_ids
            if full[:len(p_ids)] == p_ids and len(full) > len(p_ids):
                out.append(full[len(p_ids):])
            else:  # BPE merge across the boundary: score the candidate tokenized alone
                self.n_boundary_fallback += 1
                out.append(self.tok(c, add_special_tokens=False).input_ids or [self.tok.eos_token_id])
        return out

    @torch.no_grad()
    def score(self, prefix: str, cands: Sequence[str], chunk: int = 32) -> np.ndarray:
        """Sum of log-probs of each candidate continuation after `prefix`.

        One prefix forward; candidates are scored in chunks against a copy of the prefix
        KV cache (bounded memory for large candidate sets)."""
        import copy
        p_ids = self.tok(prefix, add_special_tokens=False).input_ids
        cids = self._cand_ids(p_ids, prefix, cands)
        inp = torch.tensor([p_ids], device=self.device)
        out = self.model(input_ids=inp, use_cache=True)
        first = F.log_softmax(out.logits[0, -1].float(), -1)
        res = np.array([first[c[0]].item() for c in cids], dtype=np.float64)
        P = len(p_ids)
        chunk = self._safe_chunk(chunk, P, max(len(c) for c in cids))
        for s0 in range(0, len(cids), chunk):
            sub = cids[s0:s0 + chunk]
            L = max(len(c) for c in sub)
            if L == 1:
                continue
            C = len(sub)
            x = torch.full((C, L - 1), self.tok.pad_token_id, dtype=torch.long, device=self.device)
            att = torch.zeros((C, P + L - 1), dtype=torch.long, device=self.device)
            att[:, :P] = 1
            for i, c in enumerate(sub):
                if len(c) > 1:
                    x[i, :len(c) - 1] = torch.tensor(c[:-1], device=self.device)
                    att[i, P:P + len(c) - 1] = 1
            pos = torch.arange(P, P + L - 1, device=self.device).unsqueeze(0).expand(C, -1)
            try:
                past = copy.deepcopy(out.past_key_values)
                past.batch_repeat_interleave(C)
                o2 = self.model(input_ids=x, attention_mask=att, position_ids=pos,
                                past_key_values=past, use_cache=False)
                lp = F.log_softmax(o2.logits.float(), -1)
            except Exception:  # cache API differs across versions: recompute without cache
                full = torch.cat([inp.expand(C, -1), x], 1)
                o2 = self.model(input_ids=full, attention_mask=att)
                lp = F.log_softmax(o2.logits[:, P:].float(), -1)
            for i, c in enumerate(sub):
                for t in range(1, len(c)):
                    res[s0 + i] += lp[i, t - 1, c[t]].item()
        return res

    # ------------------------------------------------------------------ L1 contrastive decoding
    @torch.no_grad()
    def generate_contrastive(self, real: List[str], cf: List[str], alpha: float = 1.0,
                             beta: float = 0.1, max_new_tokens: int = 160, gate: str = "result",
                             stop: str = "Answer:", batch_size: int = 4,
                             space: str = "prob") -> List[str]:
        """Greedy decoding with score (1+a)*p(.|real) - a*p(.|cf)  (space='prob', default)
        or (1+a)*log p(.|real) - a*log p(.|cf)  (space='log', CAD-style).

        gate='result': contrast only while emitting a hop result (text after '=>' on the
        current line); gate='all': contrast every token (CAD-style).
        Plausibility constraint: only tokens with p_real >= beta * max p_real are eligible.
        """
        outs = []
        for i in range(0, len(real), batch_size):
            outs.extend(self._contrastive_batch(real[i:i + batch_size], cf[i:i + batch_size],
                                                alpha, beta, max_new_tokens, gate, stop, space))
        return outs

    def _contrastive_batch(self, real, cf, alpha, beta, max_new_tokens, gate, stop, space):
        B = len(real)
        enc = self._enc(list(real) + list(cf))
        att = enc.attention_mask
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        out = self.model(input_ids=enc.input_ids, attention_mask=att, position_ids=pos,
                         use_cache=True)
        past, logits = out.past_key_values, out.logits[:, -1]
        last_pos = pos[:, -1]
        gen = [[] for _ in range(B)]
        text = [""] * B
        done = [False] * B
        log_beta = math.log(beta)
        eos = {self.tok.eos_token_id, self.tok.pad_token_id}
        for _ in range(max_new_tokens):
            lp = F.log_softmax(logits.float(), -1)
            nxt = []
            for b in range(B):
                if done[b]:
                    nxt.append(self.tok.pad_token_id)
                    continue
                lr, lc = lp[b], lp[B + b]
                cur_line = text[b].rsplit("\n", 1)[-1]
                on = gate == "all" or (gate == "result" and "=>" in cur_line)
                if on:
                    if space == "prob":
                        s = (1 + alpha) * lr.exp() - alpha * lc.exp()
                    else:
                        s = (1 + alpha) * lr - alpha * lc
                    s[lr < lr.max() + log_beta] = -float("inf")
                    t = int(s.argmax())
                else:
                    t = int(lr.argmax())
                gen[b].append(t)
                text[b] = self.tok.decode(gen[b], skip_special_tokens=True)
                if t in eos or stop in text[b]:
                    done[b] = True
                nxt.append(t)
            if all(done):
                break
            new = torch.tensor(nxt + nxt, device=self.device).unsqueeze(1)
            att = torch.cat([att, torch.ones((2 * B, 1), dtype=att.dtype, device=self.device)], 1)
            last_pos = last_pos + 1
            out = self.model(input_ids=new, attention_mask=att, position_ids=last_pos.unsqueeze(1),
                             past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1]
        return [t.split(stop)[0] for t in text]


In [ ]:
%%writefile hgcd/decoding.py
"""Answer-level decoding rules over a finite candidate set.

All rules consume *normalized* candidate distributions computed once per example, so any
alpha can be evaluated post hoc (cheap sweeps, no extra forward passes).

Notation (per example):
  p_cot      p(a | Q, P, y)              model's own chain y
  p_dir      p(a | Q, P)                 direct answer, no chain
  p_noctx    p(a | Q)                    no context, no chain
  p_tot[j]   p(a | Q, P, do_total(y, j))
  p_nai[j]   p(a | Q, P, do_naive(y, j))
  p_pla[j]   p(a | Q, P, placebo(y, j))
  rel        indices of DRG-relevant hops (directed path to the answer)
"""

from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np

EPS = 1e-12


def softmax_logp(lp: np.ndarray) -> np.ndarray:
    z = lp - lp.max()
    e = np.exp(z)
    return e / e.sum()


def jsd(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen-Shannon divergence in bits, in [0, 1]."""
    m = 0.5 * (p + q)

    def kl(a, b):
        return float(np.sum(a * (np.log2(a + EPS) - np.log2(b + EPS))))
    return max(0.0, min(1.0, 0.5 * kl(p, m) + 0.5 * kl(q, m)))


def _plausible(p: np.ndarray, beta: float) -> np.ndarray:
    return p >= beta * p.max()


def _argmax_masked(s: np.ndarray, mask: np.ndarray) -> int:
    s = np.where(mask, s, -np.inf)
    return int(np.argmax(s))


def log_contrast(p_pos: np.ndarray, p_neg: np.ndarray, alpha: float, beta: float) -> int:
    s = (1 + alpha) * np.log(p_pos + EPS) - alpha * np.log(p_neg + EPS)
    return _argmax_masked(s, _plausible(p_pos, beta))


def prob_contrast(p_pos: np.ndarray, p_neg: np.ndarray, alpha: float, beta: float) -> int:
    """Probability-space contrast (1+a) p - a p_neg (Prop. 1: cancels shortcut mixtures)."""
    s = (1 + alpha) * p_pos - alpha * p_neg
    return _argmax_masked(s, _plausible(p_pos, beta))


def _mix(ps: List[np.ndarray], idx: List[int]) -> Optional[np.ndarray]:
    if not idx:
        return None
    return np.mean([ps[j] for j in idx], axis=0)


METHODS = ["direct", "cot", "cad", "adacad", "cad_chain", "calib", "poe", "hgcd", "hgcd_log", "hgcd_naive",
           "hgcd_uniform", "hgcd_placebo", "hgcd_ada"]


def decide(d: Dict, method: str, alpha: float = 1.0, beta: float = 0.1) -> int:
    """Return the index of the chosen candidate for one example record `d`."""
    p_cot, p_dir, p_noctx = d["p_cot"], d["p_dir"], d["p_noctx"]
    n_hops = len(d["p_tot"])
    rel = d["rel"]
    if method == "direct":
        return int(np.argmax(p_dir))
    if method == "cot":
        return int(np.argmax(p_cot))
    if method == "sc":  # self-consistency majority vote; ties broken by p_cot
        votes = np.bincount(d["sc_votes"], minlength=len(p_cot)).astype(float)
        return int(np.argmax(votes + 1e-3 * p_cot))
    if method == "cad":
        return log_contrast(p_dir, p_noctx, alpha, beta)
    if method == "adacad":
        return log_contrast(p_dir, p_noctx, jsd(p_dir, p_noctx), beta)
    if method == "cad_chain":
        return log_contrast(p_cot, p_dir, alpha, beta)
    if method == "poe":  # two readers must agree: chain-based x direct (product of experts)
        return int(np.argmax(np.log(p_cot + EPS) + np.log(p_dir + EPS)))
    if method == "calib":  # contextual calibration (content-free input), no causal content
        return log_contrast(p_cot, p_noctx, alpha, beta)
    # ---- HGCD family: contrast the chain against do()-interventions on its own hops
    if n_hops == 0:
        return int(np.argmax(p_cot))
    if method == "hgcd":
        neg = _mix(d["p_tot"], rel)
    elif method == "hgcd_log":
        neg = _mix(d["p_tot"], rel)
        return int(np.argmax(p_cot)) if neg is None else log_contrast(p_cot, neg, alpha, beta)
    elif method == "hgcd_naive":
        neg = _mix(d["p_nai"], rel)
    elif method == "hgcd_uniform":
        neg = _mix(d["p_tot"], list(range(n_hops)))
    elif method == "hgcd_placebo":
        neg = _mix(d["p_pla"], rel)
    elif method == "hgcd_ada":
        neg = _mix(d["p_tot"], rel)
        if neg is None:
            return int(np.argmax(p_cot))
        return prob_contrast(p_cot, neg, alpha * jsd(p_cot, neg), beta)
    else:
        raise ValueError(method)
    if neg is None:
        return int(np.argmax(p_cot))
    return prob_contrast(p_cot, neg, alpha, beta)


In [ ]:
%%writefile hgcd/experiments.py
"""Experiment runners. Each runner returns plain-dict records that store the candidate
distributions, so every decoding rule / alpha is evaluated post hoc."""

from __future__ import annotations

import random
import re
import time
from typing import Dict, List, Optional

import numpy as np

from . import chain as C
from . import data as D
from .decoding import softmax_logp
from .prompts import (clutrr_messages, clutrr_question_only, musique_messages,
                      musique_question_only)


def _log(msg: str):
    print(time.strftime("[%H:%M:%S] "), msg, flush=True)


_RAW: List[np.ndarray] = []


def _dist(lm, prefix: str, cands: List[str]) -> np.ndarray:
    lp = lm.score(prefix, cands)
    _RAW[:] = [lp]           # keep only the last raw log-probs (candidate-set-free robustness)
    return softmax_logp(lp)


def _raw_last() -> np.ndarray:
    return _RAW[-1]


def _score_chain_contexts(lm, base_prompt: str, hops: List[C.Hop], ans_prefix: str,
                          cands: List[str], full: bool, marker: bool = False) -> Dict:
    """p(a | context, intervened chain) for every hop and intervention type."""
    def ctx(hs):
        return base_prompt + C.render_chain(hs) + ans_prefix
    rec = {"p_cot": _dist(lm, ctx(hops), cands)}
    rec["lp_cot"] = _raw_last()
    rec["p_tot"], rec["lp_tot"] = [], []
    for j in range(len(hops)):
        rec["p_tot"].append(_dist(lm, ctx(C.do_total(hops, j)), cands))
        rec["lp_tot"].append(_raw_last())
    if full:
        rec["p_nai"] = [_dist(lm, ctx(C.do_naive(hops, j)), cands) for j in range(len(hops))]
        rec["p_pla"] = [_dist(lm, ctx(C.placebo(hops, j)), cands) for j in range(len(hops))]
        if marker:
            rec["p_mm"] = [_dist(lm, ctx(C.mask_marker(hops, j)), cands) for j in range(len(hops))]
            rec["p_mmt"] = [_dist(lm, ctx(C.mask_marker_tree(hops, j)), cands)
                            for j in range(len(hops))]
    else:
        rec["p_nai"] = rec["p_pla"] = None
    return rec


# ============================================================================ CLUTRR

def _fixed_chain(lm, base: str, hops: List[C.Hop], ans_prefix: str, cands: List[str],
                 target=None) -> Dict:
    """Readout of a GIVEN chain under the current input: the mediator is held fixed."""
    r = _score_chain_contexts(lm, base, hops, ans_prefix, cands, full=False)
    r["rel"] = C.relevant_hops(hops, target=target)
    return r


def clutrr_score(lm, ex: D.ClutrrExample, chain_text: str, rels: List[str],
                 full: bool = True, fixed_chain: Optional[str] = None) -> Dict:
    cands = [" " + r for r in rels]
    hops = C.parse_chain(chain_text)
    base = lm.prompt(clutrr_messages(ex.story_text, ex.source, ex.target))
    rec = _score_chain_contexts(lm, base, hops, ex.answer_prefix(), cands, full)
    rec["p_dir"] = _dist(lm, base + ex.answer_prefix(), cands)
    rec["p_noctx"] = _dist(lm, lm.prompt(clutrr_question_only(ex.source, ex.target))
                           + ex.answer_prefix(), cands)
    rec["rel"] = C.relevant_hops(hops, target=ex.target)
    rec["drg_connected"] = bool(hops) and len(rec["rel"]) == len(hops)
    if fixed_chain is not None:
        # mediation: original chain M(0), with names mapped to this condition's names
        rec["fixed"] = _fixed_chain(lm, base, C.parse_chain(fixed_chain), ex.answer_prefix(),
                                    cands, target=ex.target)
    gold_tails = [h.tail for h in ex.gold_hops]
    res = [C.norm(h.result) for h in hops]
    rec.update(uid=ex.uid, k=ex.k, cond=ex.condition, gold=rels.index(ex.answer),
               n_hops=len(hops), fmt_ok=len(hops) > 0, chain=chain_text,
               hop_fid=float(np.mean([C.norm(t) in res for t in gold_tails])),
               cands=rels)
    return rec


def _gen_by_k(lm, prompts: List[str], ks: List[int], batch_size: int) -> List[str]:
    out = [None] * len(prompts)
    for k in sorted(set(ks)):
        idx = [i for i, kk in enumerate(ks) if kk == k]
        gens = lm.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 16,
                           batch_size=batch_size)
        for i, g in zip(idx, gens):
            out[i] = g
    return out


def run_clutrr(lm, examples: List[D.ClutrrExample], pool: List[D.ClutrrExample],
               rels: List[str], conditions=("orig", "rename", "order", "distract", "placebo_input"),
               batch_size: int = 8, l1: bool = False, l1_alpha: float = 1.0,
               l1_gate: str = "result", sc_n: int = 0) -> List[Dict]:
    """sc_n > 0 adds a self-consistency baseline (sc_n sampled chains, T=0.7, majority vote)
    on the 'orig' condition. It costs sc_n extra generations per question, i.e. strictly more
    compute than HGCD, which needs no extra generation."""
    recs = []
    orig_chain: Dict[str, str] = {}
    for cond in conditions:
        exs = [D.apply_condition(e, cond, pool) for e in examples]
        prompts = [lm.prompt(clutrr_messages(e.story_text, e.source, e.target)) for e in exs]
        _log(f"CLUTRR [{cond}] generating {len(exs)} chains")
        gens = _gen_by_k(lm, prompts, [e.k for e in exs], batch_size)
        _log(f"CLUTRR [{cond}] scoring")
        votes = [None] * len(exs)
        if sc_n > 0 and cond == "orig":
            _log(f"CLUTRR [sc] sampling {sc_n} chains per question")
            cands = [" " + r for r in rels]
            for k in sorted(set(e.k for e in exs)):
                idx = [i for i, e in enumerate(exs) if e.k == k]
                samples = lm.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 16,
                                      do_sample=True, n=sc_n, batch_size=max(1, batch_size // 2))
                for t, i in enumerate(idx):
                    vs = []
                    for c in samples[t * sc_n:(t + 1) * sc_n]:
                        pre = prompts[i] + C.render_chain(C.parse_chain(c)) + exs[i].answer_prefix()
                        vs.append(int(np.argmax(lm.score(pre, cands))))
                    votes[i] = vs
        for e0, e, g, v in zip(examples, exs, gens, votes):
            fixed = None
            if cond == "orig":
                orig_chain[e.uid] = g
            elif e.uid in orig_chain:
                fixed = D.map_names(orig_chain[e.uid], e0, e)
            r = clutrr_score(lm, e, g, rels, full=(cond == "orig"), fixed_chain=fixed)
            r["chain_src"] = "greedy"
            r["sc_votes"] = v
            recs.append(r)
        if l1 and cond == "orig":
            _log(f"CLUTRR [L1] contrastive generation ({l1_gate}, a={l1_alpha})")
            cf = [lm.prompt(clutrr_messages(D.counterfactual_story(e), e.source, e.target))
                  for e in exs]
            order = sorted(range(len(exs)), key=lambda i: exs[i].k)
            for s in range(0, len(order), batch_size):
                idx = order[s:s + batch_size]
                kmax = max(exs[i].k for i in idx)
                gl1 = lm.generate_contrastive([prompts[i] for i in idx], [cf[i] for i in idx],
                                              alpha=l1_alpha, gate=l1_gate,
                                              max_new_tokens=26 * (kmax + 2) + 16,
                                              batch_size=batch_size)
                for i, g in zip(idx, gl1):
                    r = clutrr_score(lm, exs[i], g, rels, full=False)
                    r["chain_src"] = "l1"
                    recs.append(r)
    return recs


# ============================================================================ estimator validation

def _rollout_pns(lm, base, pre, cands, g, corrupt, keep, k, batch_size, max_new):
    """PNS_j = P(correct | resample after original h_j) - P(correct | resample after corrupted h_j).
    Returns per-hop estimates; also stores both arms for diagnostics in `_rollout_pns.last`."""
    prompts = [base + C.render_chain(p) for p in corrupt + keep]
    conts = lm.generate(prompts, max_new_tokens=max_new, do_sample=True, n=k,
                        batch_size=max(1, batch_size // 2))
    n = len(corrupt)

    def acc(prefix_hops, outs):
        return float(np.mean([np.argmax(_dist(lm, base + C.render_chain(prefix_hops + C.parse_chain(c))
                                               + pre, cands)) == g for c in outs]))
    y_cor = [acc(corrupt[j], conts[j * k:(j + 1) * k]) for j in range(n)]
    y_keep = [acc(keep[j], conts[(n + j) * k:(n + j + 1) * k]) for j in range(n)]
    _rollout_pns.last = (y_keep, y_cor)
    return [yk - yc for yk, yc in zip(y_keep, y_cor)]


def _corrupt_clutrr_hop(h: C.Hop, rels: List[str], rng: random.Random) -> C.Hop:
    m = C.CLUTRR_HOP_RE.match(h.body)
    if not m:
        return C.Hop(h.idx, C.MASK, h.result)
    alt = rng.choice([r for r in rels if r != m.group(2)])
    return C.Hop(h.idx, f"{m.group(1)}'s {alt} is {m.group(3)}.", h.result)


def run_estimators_clutrr(lm, examples: List[D.ClutrrExample], pool: List[D.ClutrrExample],
                          rels: List[str], rollouts: int = 0, batch_size: int = 8,
                          seed: int = 0, compose=False) -> List[Dict]:
    """compose: False | True (append a hop stating the gold answer) | "wrong" (append a hop
    stating a WRONG relation that contradicts the relational hops: a reader that computes over
    the chain should resist it; a reader that copies should follow it)."""
    """Gold chains + one inserted true-but-redundant hop (true necessity 0)."""
    cands = [" " + r for r in rels]
    rows = []
    for n, ex in enumerate(examples):
        rng = random.Random(f"est-{ex.uid}-{seed}")
        ex2, red = D.with_redundant_fact(ex, pool, seed)
        hops = [C.Hop(i + 1, h.text(), h.tail) for i, h in enumerate(ex.gold_hops)]
        rpos = rng.randint(0, len(hops) - 1)
        hops.insert(rpos, C.Hop(0, red.text(), red.tail))
        wrong = None
        if compose:  # T_copy = 1: the answer becomes a string the reader can copy from the chain
            stated = ex.answer
            if compose == "wrong":
                wrong = rng.choice([r for r in rels if r != ex.answer])
                stated = wrong
            hops.append(C.Hop(len(hops) + 1, f"So {ex.target} is {ex.source}'s relative:", stated))
        base = lm.prompt(clutrr_messages(ex2.story_text, ex2.source, ex2.target))
        pre = ex2.answer_prefix()
        g = rels.index(ex.answer)
        recs = _score_chain_contexts(lm, base, hops, pre, cands, full=True, marker=True)
        p_full = recs["p_cot"]
        rel = C.relevant_hops(hops, target=ex.target) if not compose else list(range(len(hops)))
        # (b) form vs content: no chain at all vs. a chain of the same shape with every hop masked
        p_none = _dist(lm, base + pre, cands)
        p_shape = _dist(lm, base + C.render_chain([C.Hop(h.idx, C.MASK, C.MASK) for h in hops]) + pre,
                        cands)
        names = [n for n in ex2.names if n not in {h.result for h in hops}] or ex2.names
        cf_new = [rng.choice(names) for _ in hops]
        ctx = lambda hs: base + C.render_chain(hs) + pre
        p_cfd = [_dist(lm, ctx(C.replace_result(hops, j, cf_new[j], False)), cands)
                 for j in range(len(hops))]
        p_cft = [_dist(lm, ctx(C.replace_result(hops, j, cf_new[j], True)), cands)
                 for j in range(len(hops))]
        pns = [np.nan] * len(hops)
        pns_gold = [np.nan] * len(hops)
        if rollouts > 0:
            # Resampling estimator with a RESAMPLED baseline: both arms regenerate the continuation,
            # so the model's inability to regenerate a correct continuation cancels. (Comparing a
            # resampled corrupted arm with the fixed gold continuation makes every hop, including
            # provably redundant ones, look necessary.)
            corrupt = [hops[:j] + [_corrupt_clutrr_hop(hops[j], rels, rng)] for j in range(len(hops))]
            keep = [hops[:j + 1] for j in range(len(hops))]
            pns = _rollout_pns(lm, base, pre, cands, g, corrupt, keep, rollouts, batch_size,
                               26 * (len(hops) + 1) + 16)
            pns_gold = [float(np.argmax(p_full) == g) - yc for yc in _rollout_pns.last[1]]
        for j in range(len(hops)):
            rows.append(dict(
                dataset={False: "clutrr", True: "clutrr_copy", "wrong": "clutrr_copywrong"}[compose],
                uid=ex.uid, k=ex.k, hop=j,
                p_wrong_full=float(p_full[rels.index(wrong)]) if wrong else np.nan,
                follows_wrong=int(wrong is not None and np.argmax(p_full) == rels.index(wrong)),
                n_hops=len(hops), p_gold_nochain=float(p_none[g]),
                p_gold_shape=float(p_shape[g]),
                label=int(j != rpos), is_final=int(j == len(hops) - 1),
                drg_relevant=int(j in rel), p_full_gold=float(p_full[g]),
                full_correct=int(np.argmax(p_full) == g),
                d_total=float(p_full[g] - recs["p_tot"][j][g]),
                d_naive=float(p_full[g] - recs["p_nai"][j][g]),
                d_placebo=float(p_full[g] - recs["p_pla"][j][g]),
                d_maskmark=float(p_full[g] - recs["p_mm"][j][g]),
                d_maskmark_tree=float(p_full[g] - recs["p_mmt"][j][g]),
                d_total_rawlogp=float(recs["lp_cot"][g] - recs["lp_tot"][j][g]),
                d_cf_direct=float(p_full[g] - p_cfd[j][g]),
                d_cf_total=float(p_full[g] - p_cft[j][g]),
                pns_rollout=pns[j], pns_goldbase=pns_gold[j]))
        if (n + 1) % 10 == 0:
            _log(f"estimators clutrr {n + 1}/{len(examples)}")
    return rows


def _musique_cands(ex: D.MusiqueExample, extra: List[str]) -> List[str]:
    seen, out = set(), []
    for c in extra + [ex.answer] + [h[1] for h in ex.gold_hops] + \
            [p["title"] for p in ex.context()]:
        k = C.norm(c)
        if k and k not in seen:
            seen.add(k)
            out.append(c)
    return out


def run_estimators_musique(lm, examples: List[D.MusiqueExample], seed: int = 0,
                           rollouts: int = 0, batch_size: int = 4) -> List[Dict]:
    """MuSiQue gold chains (bridge-entity graphs) + an inserted redundant hop."""
    rows = []
    rng = random.Random(seed)
    for n, ex in enumerate(examples):
        other = rng.choice([o for o in examples if o.uid != ex.uid])
        hops = [C.Hop(i + 1, q, a) for i, (q, a) in enumerate(ex.gold_hops)]
        rpos = rng.randint(0, len(hops) - 1)
        hops.insert(rpos, C.Hop(0, other.gold_hops[0][0], other.gold_hops[0][1]))
        cands = [" " + c for c in _musique_cands(ex, [])]
        base = lm.prompt(musique_messages(ex.context_text(), ex.question))
        recs = _score_chain_contexts(lm, base, hops, ex.answer_prefix(), cands, full=True,
                                     marker=True)
        g = [C.norm(c) for c in cands].index(C.norm(ex.answer))
        p_full = recs["p_cot"]
        rel = C.relevant_hops(hops)
        p_none = _dist(lm, base + ex.answer_prefix(), cands)
        p_shape = _dist(lm, base + C.render_chain([C.Hop(h.idx, C.MASK, C.MASK) for h in hops])
                        + ex.answer_prefix(), cands)
        dtitles = [t for t in ex.distractor_titles() if C.norm(t) != C.norm(ex.answer)] or ["Unknown"]
        cf_new = [rng.choice(dtitles) for _ in hops]
        ctx = lambda hs: base + C.render_chain(hs) + ex.answer_prefix()
        p_cfd = [_dist(lm, ctx(C.replace_result(hops, j, cf_new[j], False)), cands)
                 for j in range(len(hops))]
        p_cft = [_dist(lm, ctx(C.replace_result(hops, j, cf_new[j], True)), cands)
                 for j in range(len(hops))]
        pns = [np.nan] * len(hops)
        pns_gold = [np.nan] * len(hops)
        if rollouts > 0:
            corrupt = [hops[:j] + [C.Hop(hops[j].idx, hops[j].body, cf_new[j])] for j in range(len(hops))]
            keep = [hops[:j + 1] for j in range(len(hops))]
            pns = _rollout_pns(lm, base, ex.answer_prefix(), cands, g, corrupt, keep, rollouts,
                               batch_size, 120)
            pns_gold = [float(np.argmax(p_full) == g) - yc for yc in _rollout_pns.last[1]]
        for j in range(len(hops)):
            rows.append(dict(
                dataset="musique", uid=ex.uid, k=ex.k, hop=j, n_hops=len(hops),
                p_gold_nochain=float(p_none[g]), p_gold_shape=float(p_shape[g]),
                label=int(j != rpos), is_final=int(j == len(hops) - 1),
                drg_relevant=int(j in rel), p_full_gold=float(p_full[g]),
                full_correct=int(np.argmax(p_full) == g),
                d_total=float(p_full[g] - recs["p_tot"][j][g]),
                d_naive=float(p_full[g] - recs["p_nai"][j][g]),
                d_placebo=float(p_full[g] - recs["p_pla"][j][g]),
                d_maskmark=float(p_full[g] - recs["p_mm"][j][g]),
                d_maskmark_tree=float(p_full[g] - recs["p_mmt"][j][g]),
                d_total_rawlogp=float(recs["lp_cot"][g] - recs["lp_tot"][j][g]),
                d_cf_direct=float(p_full[g] - p_cfd[j][g]),
                d_cf_total=float(p_full[g] - p_cft[j][g]),
                pns_rollout=pns[j], pns_goldbase=pns_gold[j]))
        if (n + 1) % 10 == 0:
            _log(f"estimators musique {n + 1}/{len(examples)}")
    return rows


# ============================================================================ MuSiQue decoding

def run_musique(lm, examples: List[D.MusiqueExample], conditions=("orig", "dist8", "order"),
                batch_size: int = 4, lean: bool = False) -> List[Dict]:
    """lean=True scores only the full chain (no per-hop interventions): enough for CoT, direct,
    CAD, calibration, PoE and the exact mediation analysis, at a fraction of the cost."""
    recs = []
    orig: Dict[str, tuple] = {}
    for cond in conditions:
        exs = [D.musique_condition(e, cond) for e in examples]
        prompts = [lm.prompt(musique_messages(e.context_text(), e.question)) for e in exs]
        _log(f"MuSiQue [{cond}] generating {len(exs)} chains")
        gens = lm.generate(prompts, max_new_tokens=200, batch_size=batch_size)
        p2 = [p + C.render_chain(C.parse_chain(g)) + "Answer:" for p, g in zip(prompts, gens)]
        free = lm.generate(p2, max_new_tokens=24, batch_size=batch_size, stop="\n")
        _log(f"MuSiQue [{cond}] scoring")
        for e, p, g, fa in zip(exs, prompts, gens, free):
            hops = C.parse_chain(g)
            fa = fa.strip().strip(".")
            cand_txt = _musique_cands_model(e, [fa] + [h.result for h in hops])
            cands = [" " + c for c in cand_txt]
            r = _score_chain_contexts(lm, p, [] if lean else hops, e.answer_prefix(), cands,
                                      full=(cond == "orig") and not lean)
            if lean:  # p_cot must still condition on the real chain
                r["p_cot"] = _dist(lm, p + C.render_chain(hops) + e.answer_prefix(), cands)
            r["sc_votes"] = None
            r["p_dir"] = _dist(lm, p + e.answer_prefix(), cands)
            r["p_noctx"] = _dist(lm, lm.prompt(musique_question_only(e.question))
                                 + e.answer_prefix(), cands)
            r["rel"] = C.relevant_hops(hops)
            if cond == "orig":
                orig[e.uid] = (hops, fa)
            elif e.uid in orig:  # mediation: original chain, intervened context
                ohops, ofa = orig[e.uid]
                fc_txt = _musique_cands_model(e, [ofa] + [h.result for h in ohops])
                fr = _fixed_chain(lm, p, [] if lean else ohops, e.answer_prefix(),
                                  [" " + c for c in fc_txt])
                if lean:
                    fr["p_cot"] = _dist(lm, p + C.render_chain(ohops) + e.answer_prefix(),
                                        [" " + c for c in fc_txt])
                fr["p_dir"] = _dist(lm, p + e.answer_prefix(), [" " + c for c in fc_txt])
                fr["p_noctx"] = _dist(lm, lm.prompt(musique_question_only(e.question))
                                      + e.answer_prefix(), [" " + c for c in fc_txt])
                fr["cands"] = fc_txt
                r["fixed"] = fr
            r.update(uid=e.uid, k=e.k, cond=cond, cands=cand_txt, gold_answer=e.answer,
                     aliases=e.aliases, free_answer=fa, n_hops=len(hops),
                     fmt_ok=len(hops) > 0, chain=g, chain_src="greedy")
            recs.append(r)
    return recs


_CAPS = re.compile(r"[A-Z][\w'\u2019\-\.]*(?:\s+(?:of|the|de|la|and|du|von|van|&)?\s*[A-Z][\w'\u2019\-\.]*)*")
_NUMS = re.compile(r"\b(?:\d{1,2}\s+)?(?:January|February|March|April|May|June|July|August|September|"
                   r"October|November|December)?\s*\d{3,4}\b|\b\d+(?:[.,]\d+)?\b")


_STOP = set("""a an the in on at for of to by from with as after before during while when where
which who whom whose what why how it its he she they his her their this that these those there here
most many some other others also however although though but and or if then thus since because
both each every all any one first last later early during despite following according""".split())


def _musique_cands_model(ex: D.MusiqueExample, model_strings: List[str], cap: int = 64) -> List[str]:
    """Candidates without access to gold: model answers and hop results first, then paragraph
    titles, then capitalised spans and numbers/dates from the context (order of appearance)."""
    pool = list(model_strings) + [p["title"] for p in ex.context()]
    for p in ex.context():
        t = p["paragraph_text"]
        pool += [m.strip() for m in _CAPS.findall(t) + _NUMS.findall(t)]
    seen, out = set(), []
    for c in pool:
        k = C.norm(c)
        if k and k not in seen and c != C.MASK and k not in _STOP:
            seen.add(k)
            out.append(c)
    return out[:cap]


In [ ]:
%%writefile hgcd/metrics.py
"""Evaluation: accuracy, EM/F1, paired effects with bootstrap CIs, McNemar, AUC."""

from __future__ import annotations

import math
import re
import string
from collections import Counter
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from .decoding import decide

GREEDY_METHODS = ["direct", "cot", "sc", "cad", "adacad", "cad_chain", "calib", "poe", "hgcd", "hgcd_log",
                  "hgcd_naive", "hgcd_uniform", "hgcd_placebo", "hgcd_ada"]
NEEDS_FULL = {"hgcd_naive", "hgcd_placebo"}


# ----------------------------------------------------------------------------- answers

def _norm_ans(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def em(pred: str, golds: Iterable[str]) -> float:
    return float(any(_norm_ans(pred) == _norm_ans(g) for g in golds))


def f1(pred: str, golds: Iterable[str]) -> float:
    best = 0.0
    pt = _norm_ans(pred).split()
    for g in golds:
        gt = _norm_ans(g).split()
        common = Counter(pt) & Counter(gt)
        ns = sum(common.values())
        if ns == 0:
            continue
        p, r = ns / len(pt), ns / len(gt)
        best = max(best, 2 * p * r / (p + r))
    return best


# ----------------------------------------------------------------------------- statistics

def boot_ci(x: np.ndarray, n: int = 2000, seed: int = 0) -> Tuple[float, float, float]:
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def mcnemar_p(a: np.ndarray, b: np.ndarray) -> float:
    """Exact two-sided McNemar test on paired binary outcomes."""
    a, b = np.asarray(a).astype(bool), np.asarray(b).astype(bool)
    n01, n10 = int(np.sum(~a & b)), int(np.sum(a & ~b))
    n = n01 + n10
    if n == 0:
        return 1.0
    k = min(n01, n10)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)


def auc(scores: np.ndarray, labels: np.ndarray) -> float:
    from sklearn.metrics import roc_auc_score
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    ok = ~np.isnan(s)
    if len(set(y[ok])) < 2:
        return np.nan
    return float(roc_auc_score(y[ok], s[ok]))


# ----------------------------------------------------------------------------- decision tables

def _word_in(word: str, text: str) -> bool:
    return re.search(r"\b" + re.escape(word.strip().lower()) + r"\b", text.lower()) is not None


def copy_table(df: pd.DataFrame) -> pd.DataFrame:
    """Copy diagnostic on the model's own chains: accuracy when the gold relation is / is not
    written somewhere in the chain, and how often the chosen answer is a word from the chain."""
    o = df[(df.cond == "orig") & df.method.isin(["direct", "cot", "hgcd", "calib"])]
    out = []
    for (m, g_in), g in o.groupby(["method", "gold_in_chain"]):
        out.append(dict(method=m, gold_in_chain=int(g_in), acc=g.correct.mean(),
                        pred_in_chain=g.pred_in_chain.mean(), n=len(g)))
    return pd.DataFrame(out)


def clutrr_table(recs: List[Dict], alpha: float = 1.0, beta: float = 0.1,
                 methods: Optional[List[str]] = None) -> pd.DataFrame:
    methods = methods or GREEDY_METHODS
    rows = []
    for r in recs:
        for m in methods:
            if r["chain_src"] == "l1" and m not in ("cot", "hgcd"):
                continue
            if m in NEEDS_FULL and r.get("p_nai") is None:
                continue
            if m == "sc" and not r.get("sc_votes"):
                continue
            i = decide(r, m, alpha, beta)
            name = m if r["chain_src"] == "greedy" else f"l1+{m}"
            chain = r.get("chain", "") or ""
            rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method=name,
                             correct=int(i == r["gold"]), pred=r["cands"][i],
                             gold_label=r["cands"][r["gold"]],
                             # copy diagnostic: is the chosen relation literally written in the chain?
                             pred_in_chain=int(_word_in(r["cands"][i], chain)),
                             gold_in_chain=int(_word_in(r["cands"][r["gold"]], chain)),
                             fmt_ok=int(r["fmt_ok"]),
                             n_hops=r["n_hops"], hop_fid=r.get("hop_fid", np.nan),
                             drg_connected=int(r.get("drg_connected", False))))
    return pd.DataFrame(rows)


def musique_table(recs: List[Dict], alpha: float = 1.0, beta: float = 0.1,
                  methods: Optional[List[str]] = None) -> pd.DataFrame:
    methods = methods or GREEDY_METHODS
    rows = []
    for r in recs:
        golds = [r["gold_answer"]] + list(r["aliases"])
        rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method="free",
                         em=em(r["free_answer"], golds), f1=f1(r["free_answer"], golds),
                         cand_recall=max(em(c, golds) for c in r["cands"])))
        for m in methods:
            if m in NEEDS_FULL and r.get("p_nai") is None:
                continue
            if m == "sc" and not r.get("sc_votes"):
                continue
            pred = r["cands"][decide(r, m, alpha, beta)]
            rows.append(dict(uid=r["uid"], k=r["k"], cond=r["cond"], method=m,
                             em=em(pred, golds), f1=f1(pred, golds),
                             cand_recall=max(em(c, golds) for c in r["cands"])))
    df = pd.DataFrame(rows)
    df["correct"] = df["em"]
    return df


def accuracy_summary(df: pd.DataFrame, by: List[str], col: str = "correct") -> pd.DataFrame:
    out = []
    for key, g in df.groupby(by):
        m, lo, hi = boot_ci(g[col].values)
        key = key if isinstance(key, tuple) else (key,)
        out.append(dict(zip(by, key), mean=m, lo=lo, hi=hi, n=len(g)))
    return pd.DataFrame(out)


def ate_table(df: pd.DataFrame, base_cond: str = "orig", col: str = "correct") -> pd.DataFrame:
    """Paired ATE of each binary intervention per method: E[Y(1) - Y(0)] (truth = 0)."""
    out = []
    base = df[df.cond == base_cond].set_index(["uid", "method"])[col]
    for cond in sorted(set(df.cond) - {base_cond}):
        cur = df[df.cond == cond].set_index(["uid", "method"])[col]
        j = pd.concat([base.rename("y0"), cur.rename("y1")], axis=1, join="inner").reset_index()
        for m, g in j.groupby("method"):
            d = (g.y1 - g.y0).values
            mean, lo, hi = boot_ci(d)
            out.append(dict(cond=cond, method=m, ate=mean, lo=lo, hi=hi, abs_ate=abs(mean),
                            n=len(g)))
    return pd.DataFrame(out)


def paired_compare(df: pd.DataFrame, a: str, b: str, cond: str = "orig",
                   col: str = "correct") -> Dict:
    x = df[(df.cond == cond) & (df.method == a)].set_index("uid")[col]
    y = df[(df.cond == cond) & (df.method == b)].set_index("uid")[col]
    j = pd.concat([x.rename("a"), y.rename("b")], axis=1, join="inner")
    mean, lo, hi = boot_ci((j.a - j.b).values)
    return dict(a=a, b=b, diff=mean, lo=lo, hi=hi, p_mcnemar=mcnemar_p(j.a.values > 0.5,
                                                                         j.b.values > 0.5),
                n=len(j))


def estimator_table(rows: List[Dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.DataFrame(rows)
    aucs = []
    for ds, g in df.groupby("dataset"):
        g2 = g[g.full_correct == 1]  # PS = 1 condition (as in PNS identifiability)
        for est in ["d_total", "d_naive", "d_cf_direct", "d_cf_total", "d_total_rawlogp", "pns_rollout", "pns_goldbase", "d_placebo",
                    "d_maskmark", "d_maskmark_tree", "drg_relevant"]:
            if est not in g or g[est].isna().all():
                continue
            aucs.append(dict(dataset=ds, estimator=est, auc_all=auc(g[est], g.label),
                             auc_ps1=auc(g2[est], g2.label), n_hops=len(g), n_ps1=len(g2)))
    df["position"] = np.where(df.label == 0, "redundant",
                              np.where(df.is_final == 1, "gold-final", "gold-nonfinal"))
    cols = [c for c in ["d_total", "d_naive", "d_cf_total", "d_cf_direct", "d_total_rawlogp",
                         "pns_rollout", "pns_goldbase", "d_maskmark_tree",
                         "d_placebo", "d_maskmark"] if c in df]
    pos = df.groupby(["dataset", "position"])[cols].mean().reset_index()
    return pd.DataFrame(aucs), pos


def fixed_records(recs: List[Dict]) -> List[Dict]:
    """Records whose chain is the ORIGINAL chain M(0) read under the intervened input."""
    out = []
    for r in recs:
        if r.get("fixed") is None or r.get("chain_src", "greedy") != "greedy":
            continue
        d = dict(r)
        d.update(r["fixed"])
        d["p_nai"] = d["p_pla"] = None
        d["sc_votes"] = None
        out.append(d)
    return out


def mediation_table(df: pd.DataFrame, df_fixed: pd.DataFrame, col: str = "correct",
                    base_cond: str = "orig") -> pd.DataFrame:
    """Decompose each intervention's total effect into a direct effect on the readout (NDE,
    chain held at M(0)) and an indirect effect through the regenerated chain (NIE = TE - NDE).
    Because the mediator is set exactly, Y(1, M(0)) is observed for every unit."""
    y0 = df[df.cond == base_cond].set_index(["uid", "method"])[col].rename("y0")
    out = []
    for cond in sorted(set(df_fixed.cond)):
        y1 = df[df.cond == cond].set_index(["uid", "method"])[col].rename("y1")
        yf = df_fixed[df_fixed.cond == cond].set_index(["uid", "method"])[col].rename("yf")
        j = pd.concat([y0, y1, yf], axis=1, join="inner").reset_index()
        for m, g in j.groupby("method"):
            te, te_lo, te_hi = boot_ci((g.y1 - g.y0).values)
            nde, nde_lo, nde_hi = boot_ci((g.yf - g.y0).values)
            nie, nie_lo, nie_hi = boot_ci((g.y1 - g.yf).values)
            out.append(dict(cond=cond, method=m, te=te, te_lo=te_lo, te_hi=te_hi, nde=nde,
                            nde_lo=nde_lo, nde_hi=nde_hi, nie=nie, nie_lo=nie_lo, nie_hi=nie_hi,
                            n=len(g)))
    return pd.DataFrame(out)


def form_content_table(rows: List[Dict]) -> pd.DataFrame:
    """Decompose the effect of writing a chain on p(gold):
    form    = p(gold | chain of the same shape, all hops masked) - p(gold | no chain)
    content = p(gold | real chain) - p(gold | masked chain of the same shape)."""
    df = pd.DataFrame(rows).drop_duplicates(["dataset", "uid"])
    if "p_gold_shape" not in df:
        return pd.DataFrame()
    df["form"] = df.p_gold_shape - df.p_gold_nochain
    df["content"] = df.p_full_gold - df.p_gold_shape
    out = []
    for ds, g in df.groupby("dataset"):
        f, flo, fhi = boot_ci(g.form.values)
        c, clo, chi = boot_ci(g.content.values)
        out.append(dict(dataset=ds, form=f, form_lo=flo, form_hi=fhi, content=c, content_lo=clo,
                        content_hi=chi, n=len(g)))
    return pd.DataFrame(out)


In [ ]:
%%writefile hgcd/report.py
"""LaTeX tables and figures written to the results directory (\\input by the paper)."""

from __future__ import annotations

import os
from typing import Dict, List

import numpy as np
import pandas as pd

from .metrics import accuracy_summary, ate_table

NAMES = {
    "direct": "Direct (no CoT)", "cot": "CoT", "sc": "Self-consistency", "cad": "CAD", "adacad": "AdaCAD",
    "cad_chain": "CAD-chain", "calib": "Calibration (content-free)", "poe": "Direct $\\times$ CoT (PoE)", "hgcd": "\\textbf{HGCD}", "hgcd_log": "HGCD (log-space)",
    "hgcd_naive": "HGCD (naive mask)", "hgcd_uniform": "HGCD (no graph)",
    "hgcd_placebo": "HGCD (placebo)", "hgcd_ada": "HGCD (adaptive $\\alpha$)",
    "l1+cot": "L1 + CoT", "l1+hgcd": "\\textbf{L1 + HGCD}", "free": "Free generation",
}
ORDER = ["direct", "cot", "sc", "cad", "adacad", "cad_chain", "calib", "poe", "hgcd_naive", "hgcd_uniform",
         "hgcd_log", "hgcd_ada", "hgcd_placebo", "hgcd", "l1+cot", "l1+hgcd", "free"]


def _fmt(m, lo, hi, pct=True):
    if m is None or np.isnan(m):
        return "--"
    s = 100 if pct else 1
    return f"{m * s:.1f}\\,{{\\scriptsize[{lo * s:.1f},{hi * s:.1f}]}}"


def _write(path: str, s: str):
    with open(path, "w") as f:
        f.write(s)


def main_table(df: pd.DataFrame, out: str, id_max_k: int = 4, col: str = "correct",
               by_split: bool = True) -> pd.DataFrame:
    d = df[df.cond == "orig"].copy()
    d["split"] = np.where(d.k <= id_max_k, "ID", "OOD")
    s2 = accuracy_summary(d, ["method"], col).assign(split="All")
    s = pd.concat([accuracy_summary(d, ["method", "split"], col), s2]) if by_split else s2
    splits = [x for x in ["ID", "OOD", "All"] if x in set(s.split)]
    lines = ["\\begin{tabular}{l" + "c" * len(splits) + "}", "\\toprule",
             "Method & " + " & ".join(f"{x} ($n$={int(s[s.split == x].n.max())})" for x in splits)
             + " \\\\", "\\midrule"]
    for m in [m for m in ORDER if m in set(s.method)]:
        cells = []
        for x in splits:
            r = s[(s.method == m) & (s.split == x)]
            cells.append(_fmt(*r[["mean", "lo", "hi"]].values[0]) if len(r) else "--")
        lines.append(f"{NAMES.get(m, m)} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")
    return s


def ate_latex(df: pd.DataFrame, out: str, col: str = "correct") -> pd.DataFrame:
    a = ate_table(df, col=col)
    conds = sorted(set(a.cond))
    lines = ["\\begin{tabular}{l" + "c" * len(conds) + "}", "\\toprule",
             "Method & " + " & ".join(c.replace("_", "\\_") for c in conds) + " \\\\",
             "\\midrule"]
    for m in [m for m in ORDER if m in set(a.method)]:
        cells = []
        for c in conds:
            r = a[(a.method == m) & (a.cond == c)]
            cells.append(_fmt(*r[["ate", "lo", "hi"]].values[0]) if len(r) else "--")
        lines.append(f"{NAMES.get(m, m)} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")
    return a


EST_NAMES = {
    "d_naive": "Mask hop only (direct)",
    "d_total": "Mask hop + DRG descendants (total)",
    "d_cf_direct": "Replace result, keep downstream (direct)",
    "d_cf_total": "Replace result, propagate (total)",
    "pns_rollout": "Corrupt + resample, resampled baseline (ours)",
    "pns_goldbase": "Corrupt + resample, gold-chain baseline (PNS as in Yu et al.)",
    "d_total_rawlogp": "Mask hop + descendants, raw log-prob (no candidate set)",
    "drg_relevant": "DRG path indicator (structure only)",
    "d_placebo": "Placebo rewrite (control)",
    "d_maskmark": "Mask token, content kept (control)",
    "d_maskmark_tree": "Mask tokens on hop + descendants, content kept (control)",
}
EST_SHORT = {"d_total": "mask-tot", "d_naive": "mask-dir", "d_cf_total": "cf-tot",
             "d_cf_direct": "cf-dir", "pns_rollout": "rollout", "pns_goldbase": "rollout-gold", "d_total_rawlogp": "tot-rawlp",
             "d_maskmark_tree": "marker-tree", "d_placebo": "placebo",
             "d_maskmark": "marker"}


def estimator_latex(aucs: pd.DataFrame, pos: pd.DataFrame, out_auc: str, out_pos: str):
    ds = sorted(set(aucs.dataset))
    lines = ["\\begin{tabular}{l" + "cc" * len(ds) + "}", "\\toprule",
             "Estimator & " + " & ".join(f"\\multicolumn{{2}}{{c}}{{{d.replace('_', chr(92) + '_')}}}"
                                         for d in ds) + " \\\\",
             " & " + " & ".join(["all", "PS$=$1"] * len(ds)) + " \\\\", "\\midrule"]
    for e in EST_NAMES:
        if e not in set(aucs.estimator):
            continue
        cells = []
        for d in ds:
            r = aucs[(aucs.dataset == d) & (aucs.estimator == e)]
            ok = len(r) and not np.isnan(r.auc_all.values[0])
            cells += ([f"{r.auc_all.values[0]:.3f}", f"{r.auc_ps1.values[0]:.3f}"] if ok else ["--", "--"])
        lines.append(f"{EST_NAMES[e]} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out_auc, "\n".join(lines) + "\n")
    cols = [c for c in EST_SHORT if c in pos.columns and not pos[c].isna().all()]
    lines = ["\\begin{tabular}{ll" + "c" * len(cols) + "}", "\\toprule",
             "Dataset & Hop type & " + " & ".join(EST_SHORT[c] for c in cols) + " \\\\", "\\midrule"]
    for _, r in pos.iterrows():
        lines.append(f"{r.dataset.replace('_', chr(92) + '_')} & {r.position} & "
                     + " & ".join(f"{r[c]:.3f}" for c in cols) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out_pos, "\n".join(lines) + "\n")


def figures(df: pd.DataFrame, sweep: pd.DataFrame, outdir: str, prefix: str = "clutrr"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = df[df.cond == "orig"]
    fig, ax = plt.subplots(figsize=(4.2, 3.0))
    for m in ["cot", "cad_chain", "hgcd_naive", "hgcd"]:
        g = d[d.method == m].groupby("k").correct.mean()
        if len(g):
            ax.plot(g.index, g.values * 100, marker="o", label=NAMES[m].replace("\\textbf{", "").replace("}", ""))
    ax.set_xlabel("number of hops $k$")
    ax.set_ylabel("accuracy (%)")
    ax.legend(fontsize=7, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"{prefix}_acc_vs_k.pdf"))
    plt.close(fig)
    if sweep is not None and len(sweep):
        fig, ax = plt.subplots(figsize=(4.2, 3.0))
        for m, g in sweep.groupby("method"):
            ax.plot(g.alpha, g.acc * 100, marker=".", label=NAMES.get(m, m).replace("\\textbf{", "").replace("}", ""))
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xlabel("$\\alpha$")
        ax.set_ylabel("accuracy (%)")
        ax.legend(fontsize=7, frameon=False)
        fig.tight_layout()
        fig.savefig(os.path.join(outdir, f"{prefix}_alpha_sweep.pdf"))
        plt.close(fig)


def readout_latex(df: pd.DataFrame, out: str, methods=("cot", "sc", "cad_chain", "hgcd_naive",
                                                      "hgcd")) -> pd.DataFrame:
    """Accuracy split by whether the greedy chain reaches every gold-path entity (hop_fid = 1).

    HGCD re-weights answers toward those the chain causally supports, so it can repair *readout*
    errors (correct chain, wrong answer) but not wrong chains.
    """
    d = df[(df.cond == "orig") & df.method.isin(methods)].copy()
    d["chain"] = np.where(d.hop_fid >= 1.0, "gold-path chain", "other chain")
    s = accuracy_summary(d, ["method", "chain"])
    groups = ["gold-path chain", "other chain"]
    lines = ["\\begin{tabular}{lcc}", "\\toprule",
             "Method & " + " & ".join(f"{g} ($n$={int(s[s.chain == g].n.max()) if (s.chain == g).any() else 0})"
                                      for g in groups) + " \\\\", "\\midrule"]
    for m in [m for m in ORDER if m in set(s.method)]:
        cells = []
        for g in groups:
            r = s[(s.method == m) & (s.chain == g)]
            cells.append(_fmt(*r[["mean", "lo", "hi"]].values[0]) if len(r) else "--")
        lines.append(f"{NAMES.get(m, m)} & " + " & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")
    return s


def mediation_latex(med: pd.DataFrame, out: str):
    lines = ["\\begin{tabular}{llccc}", "\\toprule",
             "Intervention & Method & TE & NDE (chain fixed) & NIE (via chain) \\\\", "\\midrule"]
    for cond in sorted(set(med.cond)):
        for m in [m for m in ORDER if m in set(med[med.cond == cond].method)]:
            r = med[(med.cond == cond) & (med.method == m)].iloc[0]
            lines.append(f"{cond.replace('_', chr(92) + '_')} & {NAMES.get(m, m)} & "
                         f"{_fmt(r.te, r.te_lo, r.te_hi)} & {_fmt(r.nde, r.nde_lo, r.nde_hi)} & "
                         f"{_fmt(r.nie, r.nie_lo, r.nie_hi)} \\\\")
        lines.append("\\midrule")
    lines[-1] = "\\bottomrule"
    lines.append("\\end{tabular}")
    _write(out, "\n".join(lines) + "\n")


def reliance_table(run_dirs: Dict[str, str], out_tex: str, out_pdf: str = None) -> pd.DataFrame:
    """Aggregate estimator runs across models: reliance (mean total effect of gold hops) and
    AUC of each estimator, per dataset and model."""
    from .metrics import estimator_table
    rows = []
    for label, d in run_dirs.items():
        p = os.path.join(d, "estimators.csv")
        if not os.path.exists(p):
            continue
        est = pd.read_csv(p)
        aucs, pos = estimator_table(est.to_dict("records"))
        for ds, g in est.groupby("dataset"):
            gold = g[g.label == 1]
            row = dict(model=label, dataset=ds, reliance_final=gold[gold.is_final == 1].d_total.mean(),
                       reliance_nonfinal=gold[gold.is_final == 0].d_total.mean(),
                       ps1=g.drop_duplicates("uid").full_correct.mean())
            for _, a in aucs[aucs.dataset == ds].iterrows():
                row["auc_" + a.estimator] = a.auc_all
            rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    cols = [c for c in ["auc_d_naive", "auc_d_total", "auc_d_cf_direct", "auc_d_cf_total",
                        "auc_pns_rollout", "auc_pns_goldbase", "auc_d_placebo"] if c in df]
    head = ["Model", "Data", "PS", "Rel.(final)", "Rel.(non-final)"] + \
           [EST_SHORT.get(c[4:], c[4:]) for c in cols]
    lines = ["\\begin{tabular}{ll" + "c" * (3 + len(cols)) + "}", "\\toprule",
             " & ".join(head) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        cells = [r.model, r.dataset, f"{r.ps1:.2f}", f"{r.reliance_final:.3f}",
                 f"{r.reliance_nonfinal:.3f}"] + \
                ["--" if pd.isna(r[c]) else f"{r[c]:.3f}" for c in cols]
        lines.append(" & ".join(cells) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out_tex, "\n".join(lines) + "\n")
    if out_pdf:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(4.2, 3.0))
        for ds, g in df.groupby("dataset"):
            ax.plot(g.model, g.reliance_final, marker="o", label=f"{ds} final hop")
            ax.plot(g.model, g.reliance_nonfinal, marker="s", ls="--", label=f"{ds} non-final")
        ax.set_ylabel("mean total effect on p(gold)")
        ax.legend(fontsize=7, frameon=False)
        plt.xticks(rotation=30, fontsize=7)
        fig.tight_layout()
        fig.savefig(out_pdf)
        plt.close(fig)
    return df


def form_content_latex(fc: pd.DataFrame, out: str):
    lines = ["\\begin{tabular}{lccc}", "\\toprule",
             "Dataset & Form (masked chain $-$ no chain) & Content (real $-$ masked chain) & $n$ \\\\",
             "\\midrule"]
    for _, r in fc.iterrows():
        lines.append(f"{r.dataset.replace('_', chr(92) + '_')} & {_fmt(r.form, r.form_lo, r.form_hi)} & "
                     f"{_fmt(r.content, r.content_lo, r.content_hi)} & {int(r.n)} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    _write(out, "\n".join(lines) + "\n")


In [ ]:
%%writefile hgcd/pipeline.py
"""End-to-end pipeline used by the Kaggle notebook and the CLI.

    python -m hgcd.pipeline --preset smoke      # CPU sanity run (minutes)
    python -m hgcd.pipeline --preset kaggle     # 1x T4, Qwen2.5-1.5B-Instruct (~2-4 h)
    python -m hgcd.pipeline --preset a100       # 1x A100, 7B model
"""

from __future__ import annotations

import argparse
import json
import os
import pickle
import time
from dataclasses import asdict, dataclass, field
from typing import List

import numpy as np
import pandas as pd

from . import data as D
from . import experiments as E
from . import metrics as M
from . import report as R


@dataclass
class Config:
    model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    outdir: str = "results/run"
    seed: int = 0
    batch_size: int = 8
    # CLUTRR decoding
    ks_test: List[int] = field(default_factory=lambda: [2, 3, 4, 5, 6, 7])
    n_per_k: int = 40
    n_dev: int = 60                      # validation split (k in 2..4) for alpha selection
    conditions: List[str] = field(default_factory=lambda: ["orig", "rename", "order",
                                                           "distract", "placebo_input"])
    l1: bool = True
    l1_alpha: float = 1.0
    sc_n: int = 5                        # self-consistency samples (0 disables)
    alphas: List[float] = field(default_factory=lambda: [0.0, 0.25, 0.5, 1.0, 2.0, 4.0])
    beta: float = 0.1
    # estimator validation (gold chains + inserted redundant hop)
    ks_est: List[int] = field(default_factory=lambda: [2, 3, 4])
    n_est_per_k: int = 30
    est_rollouts: int = 0
    # MuSiQue
    n_ms: int = 120
    ms_conditions: List[str] = field(default_factory=lambda: ["orig", "dist8", "order"])
    n_est_ms: int = 80
    run_musique: bool = True
    est_compose: bool = True             # copy-vs-compute: CLUTRR chains with the answer written in
    only_estimators: bool = False        # "reliance" study: estimator stage only
    run_estimators: bool = True
    run_clutrr_decoding: bool = True
    musique_lean: bool = False           # mediation-only MuSiQue run: no per-hop HGCD contexts
    device_map: str = ""                 # e.g. "auto" to shard a 7B model over 2 T4s


PRESETS = {
    "smoke": dict(model="Qwen/Qwen2.5-0.5B-Instruct", ks_test=[2, 3, 5], n_per_k=4, n_dev=6,
                  conditions=["orig", "rename", "distract"], l1=True, sc_n=3, ks_est=[2, 3],
                  n_est_per_k=4, est_rollouts=0, n_ms=6, ms_conditions=["orig", "dist8"],
                  n_est_ms=6, batch_size=4, alphas=[0.0, 0.5, 1.0, 2.0]),
    "kaggle": dict(),
    "kaggle_full": dict(n_per_k=80, n_dev=120, n_est_per_k=60, est_rollouts=4, n_ms=300,
                        n_est_ms=200),
    # Reliance-vs-scale study (cheap): run once per model size, estimator stage only.
    "reliance": dict(ks_est=[2, 3, 4], n_est_per_k=60, n_est_ms=150, est_rollouts=2,
                     only_estimators=True),
    # 7B alone (sharded over 2 T4s): no rollouts, smaller n, fits one Kaggle session
    # Mediation at larger n (reviewer W2): MuSiQue only, orig vs dist8, no per-hop contexts
    "mediation": dict(run_estimators=False, run_clutrr_decoding=False, n_ms=500,
                      ms_conditions=["orig", "dist8"], musique_lean=True, batch_size=8),
    "reliance_7b": dict(ks_est=[2, 3, 4], n_est_per_k=40, n_est_ms=100, est_rollouts=0,
                        only_estimators=True, batch_size=4),
    "a100": dict(model="Qwen/Qwen2.5-7B-Instruct", ks_test=[2, 3, 4, 5, 6, 7, 8, 9, 10],
                 n_per_k=100, n_dev=150, n_est_per_k=80, est_rollouts=4, n_ms=500,
                 n_est_ms=300, batch_size=16),
}


def _log(m):
    print(time.strftime("[%H:%M:%S] "), m, flush=True)


def _cached(path: str, fn):
    """Stage-level checkpoint: reuse a finished stage after a crash or session restart."""
    if os.path.exists(path):
        _log(f"reusing cached stage {os.path.basename(path)}")
        with open(path, "rb") as f:
            return pickle.load(f)
    out = fn()
    with open(path, "wb") as f:
        pickle.dump(out, f)
    return out


def select_alpha(recs, methods, alphas, beta, table_fn):
    rows = []
    for a in alphas:
        df = table_fn(recs, alpha=a, beta=beta, methods=methods)
        for m, g in df.groupby("method"):
            rows.append(dict(method=m, alpha=a, acc=g.correct.mean()))
    sweep = pd.DataFrame(rows)
    best = {m: float(g.sort_values(["acc", "alpha"], ascending=[False, True]).alpha.iloc[0])
            for m, g in sweep.groupby("method")}
    return best, sweep


def _table_with_alphas(recs, best, beta, table_fn):
    parts = []
    for m, a in best.items():
        parts.append(table_fn(recs, alpha=a, beta=beta, methods=[m.replace("l1+", "")]))
    df = pd.concat(parts)
    keep = set(best)
    return df[df.method.isin(keep)].drop_duplicates()


def run(cfg: Config):
    from .lm import LM
    os.makedirs(cfg.outdir, exist_ok=True)
    with open(os.path.join(cfg.outdir, "config.json"), "w") as f:
        json.dump(asdict(cfg), f, indent=1)
    t0 = time.time()
    lm = LM(cfg.model, device_map=cfg.device_map or None)
    _log(f"loaded {cfg.model} on {lm.device}")
    summary = {"model": cfg.model}

    # ------------------------------------------------------------- data
    train = D.load_clutrr("train")
    test = D.load_clutrr("test")
    dev = D.load_clutrr("validation")
    rels = D.clutrr_relations(train + test)
    pool = train
    summary["clutrr_filter"] = {"train": len(train), "test": len(test), "dev": len(dev)}

    ms = D.load_musique("dev") if cfg.run_musique else None
    if cfg.run_estimators:
        # ------------------------------------------------------------- 1) estimator validation
        est_ex = D.sample_by_k(test, cfg.ks_est, cfg.n_est_per_k, cfg.seed + 7)
        _log(f"estimator validation on {len(est_ex)} CLUTRR gold chains")
        est_rows = _cached(os.path.join(cfg.outdir, "stage_est_clutrr.pkl"),
                           lambda: E.run_estimators_clutrr(lm, est_ex, pool, rels,
                                                           rollouts=cfg.est_rollouts,
                                                           batch_size=cfg.batch_size, seed=cfg.seed))
        if cfg.est_compose:
            _log("estimator validation on CLUTRR chains + explicit answer hop (copy vs compute)")
            est_rows = est_rows + _cached(os.path.join(cfg.outdir, "stage_est_clutrr_copy.pkl"),
                                          lambda: E.run_estimators_clutrr(lm, est_ex, pool, rels,
                                                                          batch_size=cfg.batch_size,
                                                                          seed=cfg.seed, compose=True))
            est_rows = est_rows + _cached(os.path.join(cfg.outdir, "stage_est_clutrr_copywrong.pkl"),
                                          lambda: E.run_estimators_clutrr(lm, est_ex, pool, rels,
                                                                          batch_size=cfg.batch_size,
                                                                          seed=cfg.seed, compose="wrong"))
            er = pd.DataFrame(est_rows)
            cw = er[er.dataset == "clutrr_copywrong"].drop_duplicates("uid").set_index("uid")
            ps = er[er.dataset == "clutrr"].drop_duplicates("uid").set_index("uid").full_correct
            cw = cw.join(ps.rename("ps_plain"), how="left")
            comp = cw[cw.ps_plain == 1]   # chains the model DOES solve from the relational hops alone
            summary["copy_wrong"] = dict(
                follow_rate=float(cw.follows_wrong.mean()), n=int(len(cw)),
                follow_rate_when_solvable=float(comp.follows_wrong.mean()) if len(comp) else None,
                n_solvable=int(len(comp)), p_gold=float(cw.p_full_gold.mean()))
            _log(f"wrong answer hop: followed in {cw.follows_wrong.mean():.2f} of chains; "
                 f"in {comp.follows_wrong.mean() if len(comp) else float('nan'):.2f} of the "
                 f"{len(comp)} chains the model solves without it")
        if cfg.run_musique:
            rng = np.random.default_rng(cfg.seed)
            ms_est = [ms[i] for i in rng.choice(len(ms), cfg.n_est_ms, replace=False)]
            _log(f"estimator validation on {len(ms_est)} MuSiQue gold chains")
            est_rows = est_rows + _cached(os.path.join(cfg.outdir, "stage_est_musique.pkl"),
                                          lambda: E.run_estimators_musique(lm, ms_est, seed=cfg.seed,
                                                                           rollouts=cfg.est_rollouts,
                                                                           batch_size=cfg.batch_size))
        pd.DataFrame(est_rows).to_csv(os.path.join(cfg.outdir, "estimators.csv"), index=False)
        aucs, pos = M.estimator_table(est_rows)
        R.estimator_latex(aucs, pos, os.path.join(cfg.outdir, "tab_estimators_auc.tex"),
                          os.path.join(cfg.outdir, "tab_estimators_pos.tex"))
        summary["estimator_auc"] = aucs.to_dict("records")
        summary["estimator_position"] = pos.to_dict("records")
        fc = M.form_content_table(est_rows)
        if len(fc):
            R.form_content_latex(fc, os.path.join(cfg.outdir, "tab_form_content.tex"))
            summary["form_content"] = fc.to_dict("records")
            _log("\n" + fc.to_string())
        _log("\n" + aucs.to_string())
        _log("\n" + pos.to_string())
    if cfg.only_estimators:
        summary["minutes"] = (time.time() - t0) / 60
        with open(os.path.join(cfg.outdir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=1, default=float)
        _log(f"done (estimators only) in {summary['minutes']:.1f} min -> {cfg.outdir}")
        return summary

    best = {m: 1.0 for m in M.GREEDY_METHODS}
    if cfg.run_clutrr_decoding:
        # ------------------------------------------------------------- 2) alpha selection on dev
        dev_ex = D.sample_by_k(dev, [2, 3, 4], max(1, cfg.n_dev // 3), cfg.seed + 1)
        _log(f"alpha selection on {len(dev_ex)} CLUTRR validation examples")
        dev_recs = _cached(os.path.join(cfg.outdir, "stage_dev.pkl"),
                           lambda: E.run_clutrr(lm, dev_ex, pool, rels, conditions=["orig"],
                                                batch_size=cfg.batch_size, l1=False))
        best, sweep = select_alpha(dev_recs, M.GREEDY_METHODS, cfg.alphas, cfg.beta, M.clutrr_table)
        for m in M.GREEDY_METHODS:          # methods absent on dev (e.g. sc) need no alpha
            best.setdefault(m, 0.0)
        best["l1+cot"], best["l1+hgcd"] = 0.0, best.get("hgcd", 1.0)
        sweep.to_csv(os.path.join(cfg.outdir, "alpha_sweep_dev.csv"), index=False)
        summary["alpha"] = best
        _log(f"selected alphas: {best}")

        # ------------------------------------------------------------- 3) CLUTRR test
        test_ex = D.sample_by_k(test, cfg.ks_test, cfg.n_per_k, cfg.seed)
        _log(f"CLUTRR test: {len(test_ex)} examples x {len(cfg.conditions)} conditions")
        recs = _cached(os.path.join(cfg.outdir, "clutrr_records.pkl"),
                       lambda: E.run_clutrr(lm, test_ex, pool, rels, conditions=cfg.conditions,
                                            batch_size=cfg.batch_size, l1=cfg.l1,
                                            l1_alpha=cfg.l1_alpha, sc_n=cfg.sc_n))
        df = _table_with_alphas(recs, best, cfg.beta, M.clutrr_table)
        df.to_csv(os.path.join(cfg.outdir, "clutrr_decisions.csv"), index=False)
        s = R.main_table(df, os.path.join(cfg.outdir, "tab_clutrr_main.tex"))
        a = R.ate_latex(df[~df.method.str.startswith("l1+")],
                        os.path.join(cfg.outdir, "tab_clutrr_ate.tex"))
        R.figures(df, sweep, cfg.outdir, "clutrr")
        ro = R.readout_latex(df, os.path.join(cfg.outdir, "tab_clutrr_readout.tex"))
        summary["clutrr_readout"] = ro.to_dict("records")
        cp = M.copy_table(df)
        if len(cp):
            cp.to_csv(os.path.join(cfg.outdir, "clutrr_copy_diagnostic.csv"), index=False)
            summary["clutrr_copy_diagnostic"] = cp.to_dict("records")
            _log("\n" + cp.to_string())
        fx = M.fixed_records(recs)
        if fx:
            dff = _table_with_alphas(fx, {m: a for m, a in best.items() if not m.startswith("l1+")},
                                     cfg.beta, M.clutrr_table)
            med = M.mediation_table(df, dff)
            R.mediation_latex(med, os.path.join(cfg.outdir, "tab_clutrr_mediation.tex"))
            summary["clutrr_mediation"] = med.to_dict("records")
            _log("\n" + med[["cond", "method", "te", "nde", "nie"]].to_string())
        comps = [M.paired_compare(df, "hgcd", b) for b in ["cot", "sc", "cad_chain", "calib", "hgcd_placebo",
                                                            "hgcd_naive", "hgcd_uniform", "hgcd_log"]]
        if cfg.l1:
            comps.append(M.paired_compare(df, "l1+hgcd", "cot"))
        summary["clutrr_acc"] = s.to_dict("records")
        summary["clutrr_ate"] = a.to_dict("records")
        summary["clutrr_paired"] = comps
        o = df[df.cond == "orig"]
        summary["clutrr_format"] = {"fmt_ok": float(o.fmt_ok.mean()), "hop_fid": float(o.hop_fid.mean()),
                                    "drg_connected": float(o.drg_connected.mean()),
                                    "mean_hops": float(o.n_hops.mean())}
        _log("\n" + s.to_string())
        _log("\n" + pd.DataFrame(comps).to_string())

    # ------------------------------------------------------------- 4) MuSiQue
    if cfg.run_musique:
        rng = np.random.default_rng(cfg.seed + 3)
        ms_ex = [ms[i] for i in rng.choice(len(ms), cfg.n_ms, replace=False)]
        _log(f"MuSiQue: {len(ms_ex)} examples x {len(cfg.ms_conditions)} conditions")
        mrecs = _cached(os.path.join(cfg.outdir, "musique_records.pkl"),
                        lambda: E.run_musique(lm, ms_ex, conditions=cfg.ms_conditions,
                                              batch_size=max(1, cfg.batch_size // 2),
                                              lean=cfg.musique_lean))
        mbest = {m: best.get(m, 1.0) for m in M.GREEDY_METHODS}
        mdf = _table_with_alphas(mrecs, mbest, cfg.beta, M.musique_table)
        free = M.musique_table(mrecs, methods=["cot"])
        mdf = pd.concat([mdf, free[free.method == "free"]])
        mdf.to_csv(os.path.join(cfg.outdir, "musique_decisions.csv"), index=False)
        ms_s = R.main_table(mdf, os.path.join(cfg.outdir, "tab_musique_main.tex"), col="em",
                            by_split=False)
        ms_a = R.ate_latex(mdf, os.path.join(cfg.outdir, "tab_musique_ate.tex"), col="em")
        mfx = M.fixed_records(mrecs)
        if mfx:
            mdff = _table_with_alphas(mfx, mbest, cfg.beta, M.musique_table)
            mmed = M.mediation_table(mdf, mdff, col="em")
            R.mediation_latex(mmed, os.path.join(cfg.outdir, "tab_musique_mediation.tex"))
            summary["musique_mediation"] = mmed.to_dict("records")
            _log("\n" + mmed[["cond", "method", "te", "nde", "nie"]].to_string())
        summary["musique_em"] = ms_s.to_dict("records")
        summary["musique_ate"] = ms_a.to_dict("records")
        summary["musique_cand_recall"] = float(mdf[mdf.cond == "orig"].cand_recall.mean())
        summary["musique_paired"] = [M.paired_compare(mdf, "hgcd", b, col="em")
                                     for b in ["cot", "cad", "cad_chain", "hgcd_placebo", "hgcd_naive"]]
        _log("\n" + ms_s.to_string())

    summary["boundary_fallbacks"] = lm.n_boundary_fallback
    summary["minutes"] = (time.time() - t0) / 60
    with open(os.path.join(cfg.outdir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=1, default=float)
    _log(f"done in {summary['minutes']:.1f} min -> {cfg.outdir}")
    return summary


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preset", default="smoke", choices=list(PRESETS))
    ap.add_argument("--model", default=None)
    ap.add_argument("--outdir", default=None)
    ap.add_argument("--device_map", default="")
    args = ap.parse_args()
    cfg = Config(**PRESETS[args.preset])
    if args.model:
        cfg.model = args.model
    cfg.outdir = args.outdir or f"results/{args.preset}"
    cfg.device_map = args.device_map
    run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hgcd/causaldec.py
"""Decoding that injects causal / structural information *before or during* chain generation.

Motivation (paper, R3-R5): answers are read from the chain, not computed over it, and distractor
harm is mediated by the regenerated chain. Readout-level contrast cannot help, so the methods here
act on the two places the causal analysis points to:

CLUTRR  (make the answer a deterministic function of the chain, i.e. force use of the chain)
  sym        parse the model's own hop chain into (head, relation, tail) triples and compose the
             relations with a kinship composition table induced from the *training* split
  sym_sc     the same over the sampled chains (compute-matched with sc), majority vote over valid
             compositions, falling back to sc when no sampled chain composes
MuSiQue (cut the spurious path distractor -> chain before the chain is generated)
  attn_sel   rank paragraphs by the model's own attention from the question to each paragraph,
             calibrated by a content-free question (ICR-style), keep the top K, then CoT
  self_sel   the model lists the relevant paragraph numbers, keep those, then CoT
  last_hop   answer = result of the chain's last hop (answer depends on the chain by construction)

Baselines: greedy CoT with free-form answer (cot_free), self-consistency over sampled chains (sc),
label scoring after CoT on CLUTRR (cot_score); oracle ceilings (sym_gold, gold_only) are reported
but are not methods.

    python -m hgcd.causaldec --task clutrr  --model Qwen/Qwen2.5-3B-Instruct --outdir out/c3b
    python -m hgcd.causaldec --task musique --model Qwen/Qwen2.5-3B-Instruct --outdir out/m3b
"""

from __future__ import annotations

import argparse
import copy
import json
import os
import re
import time
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from . import chain as C
from . import data as D
from . import metrics as M
from .prompts import clutrr_messages, musique_messages

T0 = time.time()


def _log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================================ CLUTRR: composition

def compose_counts(table: Dict[Tuple[str, str], str], rels: Tuple[str, ...]) -> Counter:
    """Results of composing `rels` over every binary bracketing (count = #bracketings)."""
    n = len(rels)
    memo: Dict[Tuple[int, int], Counter] = {}

    def f(i, j):
        if (i, j) in memo:
            return memo[(i, j)]
        if j - i == 1:
            out = Counter({rels[i]: 1})
        else:
            out = Counter()
            for m in range(i + 1, j):
                for a, ca in f(i, m).items():
                    for b, cb in f(m, j).items():
                        r = table.get((a, b))
                        if r is not None:
                            out[r] += ca * cb
        memo[(i, j)] = out
        return out
    return f(0, n) if n else Counter()


def compose(table, rels) -> Optional[str]:
    c = compose_counts(table, tuple(rels))
    return c.most_common(1)[0][0] if c else None


def induce_table(train: List[D.ClutrrExample], iters: int = 8, purity: float = 0.9) -> Dict:
    """Pairwise composition rules (r1, r2) -> r induced from training stories only: a split of a
    training chain whose two halves each compose to a single relation yields one observation."""
    table: Dict[Tuple[str, str], str] = {}
    for _ in range(iters):
        obs = defaultdict(Counter)
        for e in train:
            rels = tuple(h.rel for h in e.gold_hops)
            for m in range(1, len(rels)):
                left, right = compose_counts(table, rels[:m]), compose_counts(table, rels[m:])
                if len(left) == 1 and len(right) == 1:
                    obs[(next(iter(left)), next(iter(right)))][e.answer] += 1
        new = {k: v.most_common(1)[0][0] for k, v in obs.items()
               if v.most_common(1)[0][1] >= purity * sum(v.values())}
        if all(table.get(k) == r for k, r in new.items()):
            break
        table.update(new)
    return table


_HOP_REL = re.compile(r"^\s*(.+?)['’]s?\s+([A-Za-z][A-Za-z\- ]*?)\s+is\s+(.+?)\s*\.?\s*$")
# "X has a <relation> named Y."
_HOP_HAS = re.compile(r"^\s*([A-Z][\w\-]*)\s+has\s+an?\s+([A-Za-z][A-Za-z\- ]*?)\s+(?:named|called)\s+"
                      r"([A-Z][\w\-]*)\s*\.?\s*$")
# the same fact written the other way round: "Y is X's <relation>."
_HOP_REL_INV = re.compile(r"^\s*([A-Z][\w\-]*(?:\s+[A-Z][\w\-]*)?)\s+is\s+(.+?)['’]s?\s+"
                          r"([A-Za-z][A-Za-z\- ]*?)\s*\.?\s*$")


def _rel_norm(r: str) -> str:
    r = r.lower().strip()
    r = re.sub(r"\s*-\s*", "-", r)
    r = re.sub(r"\s+in\s+law$", "-in-law", r)
    r = r.replace("grand ", "grand")
    return r


def parse_triples(chain_text: str) -> List[Tuple[str, str, str]]:
    out = []
    for h in C.parse_chain(chain_text):
        m = _HOP_REL.match(h.body)
        mi = _HOP_REL_INV.match(h.body)
        mh = _HOP_HAS.match(h.body)
        if mh:
            out.append((mh.group(1).strip(), _rel_norm(mh.group(2)), mh.group(3).strip()))
        elif mi:
            out.append((mi.group(2).strip(), _rel_norm(mi.group(3)), mi.group(1).strip()))
        elif m:
            out.append((m.group(1).strip(), _rel_norm(m.group(2)), m.group(3).strip().strip(".")))
        else:
            out.append(("", "", ""))
    return out


def chain_compose(table, triples, src: str, tgt: str, vocab: set) -> Tuple[Optional[str], str]:
    """Answer computed from the chain alone, or None with the reason it is not computable."""
    if not triples:
        return None, "no_hops"
    if any(not t[1] for t in triples):
        return None, "unparsed"
    if any(t[1] not in vocab for t in triples):
        return None, "rel_oov"
    n = C.norm
    if n(triples[0][0]) != n(src):
        return None, "bad_start"
    for a, b in zip(triples, triples[1:]):
        if n(a[2]) != n(b[0]):
            return None, "broken"
    if n(triples[-1][2]) != n(tgt):
        return None, "bad_end"
    r = compose(table, [t[1] for t in triples])
    return (r, "ok") if r else (None, "no_rule")


_FEMALE = {"mother", "daughter", "sister", "wife", "grandmother", "granddaughter", "aunt", "niece",
           "mother-in-law", "daughter-in-law"}
# inverse relation by the gender of the person it now describes: (rel) -> (male, female)
_INV = {"father": ("son", "daughter"), "mother": ("son", "daughter"),
        "son": ("father", "mother"), "daughter": ("father", "mother"),
        "brother": ("brother", "sister"), "sister": ("brother", "sister"),
        "husband": ("husband", "wife"), "wife": ("husband", "wife"),
        "grandfather": ("grandson", "granddaughter"), "grandmother": ("grandson", "granddaughter"),
        "grandson": ("grandfather", "grandmother"), "granddaughter": ("grandfather", "grandmother"),
        "uncle": ("nephew", "niece"), "aunt": ("nephew", "niece"),
        "nephew": ("uncle", "aunt"), "niece": ("uncle", "aunt"),
        "father-in-law": ("son-in-law", "daughter-in-law"), "mother-in-law": ("son-in-law", "daughter-in-law"),
        "son-in-law": ("father-in-law", "mother-in-law"),
        "daughter-in-law": ("father-in-law", "mother-in-law")}


def graph_compose(table, triples, src: str, tgt: str, vocab: set,
                  gender_prior: Optional[Dict[str, str]] = None) -> Tuple[Optional[str], str]:
    """Treat every fact the model wrote as an edge of a kinship graph (order-free), add inverse
    edges when the gender they need is stated by the model's own relation words, find the shortest
    path src -> tgt and compose its relations with the rule table."""
    n = C.norm
    facts = [(n(a), r, n(b)) for a, r, b in triples if r in vocab and a and b and n(a) != n(b)]
    if not facts:
        return None, "no_facts"
    gender = dict(gender_prior or {})       # 'm' / 'f' by normalised name; stated facts override
    for a, r, b in facts:
        gender[b] = "f" if r in _FEMALE else "m"
    adj = defaultdict(list)
    for a, r, b in facts:
        adj[a].append((r, b))
        if a in gender and r in _INV:      # "a's r is b"  =>  "b's r' is a", r' gendered by a
            adj[b].append((_INV[r][gender[a] == "f"], a))
    s, t = n(src), n(tgt)
    prev, frontier = {s: None}, [s]
    while frontier and t not in prev:
        nxt = []
        for u in frontier:
            for r, v in adj[u]:
                if v not in prev:
                    prev[v] = (u, r)
                    nxt.append(v)
        frontier = nxt
    if t not in prev:
        return None, "no_path"
    rels, v = [], t
    while prev[v] is not None:
        u, r = prev[v]
        rels.append(r)
        v = u
    r = compose(table, rels[::-1])
    return (r, "ok") if r else (None, "no_rule")


EXTRACT_SYSTEM = ("List every family relation stated in the story, one per line, in the form "
                  "\"X's <relation> is Y.\" Use only these relation words: {rels}. "
                  "Write only facts the story states; do not infer or answer anything.")
EXTRACT_SHOTS = [
    ("Mabel took her son Rupert to the zoo. Rupert and his sister Agatha baked a cake for their "
     "grandmother.", ["Mabel's son is Rupert.", "Rupert's sister is Agatha."]),
    ("Clement went fishing with his father Bertram. Bertram called his sister Winifred to wish her a "
     "happy birthday. Winifred was proud of her daughter Odette.",
     ["Clement's father is Bertram.", "Bertram's sister is Winifred.", "Winifred's daughter is Odette."]),
]


EXTRACT_GENDER_NOTE = (" Then write one last line 'Genders:' giving every person's gender as stated "
                       "by the story's pronouns, e.g. 'Genders: Ann=female, Bob=male'.")
EXTRACT_SHOT_GENDERS = ["Genders: Mabel=female, Rupert=male, Agatha=female",
                        "Genders: Clement=male, Bertram=male, Winifred=female, Odette=female"]


def extract_messages(story: str, rels: List[str], genders: bool = False) -> List[dict]:
    sysm = EXTRACT_SYSTEM.format(rels=", ".join(rels)) + (EXTRACT_GENDER_NOTE if genders else "")
    msgs = [{"role": "system", "content": sysm}]
    for (st, facts), gl in zip(EXTRACT_SHOTS, EXTRACT_SHOT_GENDERS):
        msgs.append({"role": "user", "content": f"Story: {st}"})
        msgs.append({"role": "assistant", "content": "\n".join(facts + ([gl] if genders else []))})
    msgs.append({"role": "user", "content": f"Story: {story}"})
    return msgs


def parse_genders(text: Optional[str]) -> Dict[str, str]:
    out = {}
    for ln in (text or "").splitlines():
        if ln.strip().lower().startswith("genders"):
            for name, g in re.findall(r"([A-Z][\w\-]*)\s*[=:]\s*(male|female|man|woman|boy|girl)", ln, re.I):
                out[C.norm(name)] = "f" if g.lower() in ("female", "woman", "girl") else "m"
    return out


def induce_gender_lexicon(train: List[D.ClutrrExample], purity: float = 0.95) -> Dict[str, str]:
    """Name -> gender learned from the training split only (CLUTRR reuses a fixed name pool)."""
    lex = defaultdict(Counter)
    for e in train:
        for k, v in e.genders.items():
            lex[C.norm(k)][v] += 1
    return {k: ("f" if c.most_common(1)[0][0] == "female" else "m") for k, c in lex.items()
            if c.most_common(1)[0][1] >= purity * sum(c.values())}


def parse_facts(text: Optional[str]) -> List[Tuple[str, str, str]]:
    """Facts written one per line (no 'Hop i:' prefix needed)."""
    lines = [ln.strip().lstrip("-*0123456789. ").strip() for ln in (text or "").splitlines()]
    return parse_triples("".join(f"Hop {i + 1}: {ln} => x\n" for i, ln in enumerate(lines) if ln))


def _first_rel(text: str, vocab: List[str]) -> Optional[str]:
    t = " " + text.lower() + " "
    best = None
    for r in vocab:
        m = re.search(r"(?<![a-z\-])" + re.escape(r) + r"(?![a-z\-])", t)
        if m and (best is None or m.start() < best[0] or (m.start() == best[0] and len(r) > len(best[1]))):
            best = (m.start(), r)
    return best[1] if best else None


def _vote(xs: List[Optional[str]], key=lambda s: s) -> Optional[str]:
    xs = [x for x in xs if x]
    if not xs:
        return None
    c = Counter(key(x) for x in xs)
    top = c.most_common(1)[0][0]
    return next(x for x in xs if key(x) == top)


def run_clutrr(lm, args) -> pd.DataFrame:
    train, test = D.load_clutrr("train"), D.load_clutrr("test")
    table = induce_table(train)
    vocab_l = D.clutrr_relations(train)
    vocab = set(vocab_l)
    _log(f"composition table: {len(table)} rules from {len(train)} training stories")
    glex = induce_gender_lexicon(train)
    pool = (test if (args.split == "test" and args.task_name == D.CLUTRR_TASK)
            else D.load_clutrr(args.split, task=args.task_name))
    ex = D.sample_by_k(pool, args.ks, args.n_per_k, args.seed)
    cov = np.mean([C.norm(n) in glex for e in ex for n in e.names])
    _log(f"gender lexicon: {len(glex)} names from train; covers {cov:.2f} of people in this sample")
    _log(f"CLUTRR test: {len(ex)} examples, k in {sorted(set(e.k for e in ex))}")
    rows = []
    for s in range(0, len(ex), args.chunk):
        if (time.time() - T0) / 60 > args.max_minutes:
            _log("time budget reached; stopping early")
            break
        part = ex[s:s + args.chunk]
        prompts = [lm.prompt(clutrr_messages(e.story_text, e.source, e.target)) for e in part]
        mx = [26 * (e.k + 2) + 16 for e in part]
        greedy, samples, extra_s = [None] * len(part), [None] * len(part), [None] * len(part)
        nsm = args.sc_n + args.sc_extra if args.sc_n else 0   # extra samples: compute-matched SC
        for k in sorted(set(e.k for e in part)):            # batch by k: similar lengths
            idx = [i for i, e in enumerate(part) if e.k == k]
            g = lm.generate([prompts[i] for i in idx], max_new_tokens=mx[idx[0]], batch_size=args.bs)
            sm = lm.generate([prompts[i] for i in idx], max_new_tokens=mx[idx[0]], do_sample=True,
                             n=nsm, batch_size=max(1, args.bs // 2)) if nsm else []
            for t, i in enumerate(idx):
                greedy[i] = g[t]
                samples[i] = sm[t * nsm:t * nsm + args.sc_n] if nsm else []
                extra_s[i] = sm[t * nsm + args.sc_n:(t + 1) * nsm] if nsm else []
        # fact extraction (no reasoning asked): greedy + sampled, then graph search + rule table
        ext_g, ext_s = [None] * len(part), [None] * len(part)
        if args.extract:
            ep = [lm.prompt(extract_messages(e.story_text, vocab_l, genders=bool(args.extract_genders)))
                  for e in part]
            mxe = [20 * (e.k + 3) for e in part]
            for k in sorted(set(e.k for e in part)):
                idx = [i for i, e in enumerate(part) if e.k == k]
                g = lm.generate([ep[i] for i in idx], max_new_tokens=mxe[idx[0]], batch_size=args.bs,
                                stop="\n\n")
                sm = lm.generate([ep[i] for i in idx], max_new_tokens=mxe[idx[0]], do_sample=True,
                                 n=args.sc_n, batch_size=max(1, args.bs // 2), stop="\n\n") if args.sc_n else []
                for t, i in enumerate(idx):
                    ext_g[i] = g[t]
                    ext_s[i] = sm[t * args.sc_n:(t + 1) * args.sc_n] if args.sc_n else []
        # ablation: same extracted facts (and genders), but the LLM composes instead of the rule table
        llm_ext = [None] * len(part)
        if args.extract and args.ext_llm:
            fp = [lm.prompt(clutrr_messages(ext_g[i] or "(no facts)", e.source, e.target))
                  for i, e in enumerate(part)]
            fch = lm.generate(fp, max_new_tokens=26 * (max(e.k for e in part) + 2) + 16, batch_size=args.bs)
            fans = lm.generate([p_ + C.render_chain(C.parse_chain(c_)) + e.answer_prefix()
                                for p_, c_, e in zip(fp, fch, part)], max_new_tokens=8,
                               batch_size=args.bs * 2, stop="\n")
            llm_ext = [_first_rel(a_, vocab_l) for a_ in fans]
        # free-form answer after each chain (greedy and sampled), one batched pass
        ans_prompts, owner = [], []
        for i, e in enumerate(part):
            for j, ch in enumerate([greedy[i]] + samples[i] + extra_s[i]):
                ans_prompts.append(prompts[i] + C.render_chain(C.parse_chain(ch)) + e.answer_prefix())
                owner.append((i, j))
        free = lm.generate(ans_prompts, max_new_tokens=8, batch_size=args.bs * 2, stop="\n")
        free_by = defaultdict(dict)
        for (i, j), f in zip(owner, free):
            free_by[i][j] = _first_rel(f, vocab_l)
        for i, e in enumerate(part):
            golds = [e.answer]
            chains = [greedy[i]] + samples[i]
            sym = [chain_compose(table, parse_triples(c), e.source, e.target, vocab) for c in chains]
            score_pre = prompts[i] + C.render_chain(C.parse_chain(greedy[i])) + e.answer_prefix()
            cot_score = vocab_l[int(np.argmax(lm.score(score_pre, [" " + r for r in vocab_l])))]
            gold_sym = compose(table, [h.rel for h in e.gold_hops])
            trip = parse_triples(greedy[i])
            gold_t = {(C.norm(h.head), h.rel, C.norm(h.tail)) for h in e.gold_hops}
            fid = np.mean([(C.norm(a), r, C.norm(b)) in gold_t for a, r, b in trip]) if trip else 0.0
            gsym = [graph_compose(table, parse_triples(c), e.source, e.target, vocab) for c in chains]
            sc_vote = _vote([free_by[i][j] for j in range(1, len(chains))]) if samples[i] else free_by[i][0]
            preds_ext = {}
            if args.extract:
                ef = [parse_facts(x) for x in [ext_g[i]] + ext_s[i]]
                gm = [parse_genders(x) for x in [ext_g[i]] + ext_s[i]]
                ex_res = [graph_compose(table, f, e.source, e.target, vocab) for f in ef]
                lex_res = [graph_compose(table, f, e.source, e.target, vocab, glex) for f in ef]
                g_res = [graph_compose(table, f, e.source, e.target, vocab, g) for f, g in zip(ef, gm)]
                gl_res = [graph_compose(table, f, e.source, e.target, vocab, {**glex, **g})
                          for f, g in zip(ef, gm)]
                union = [t for f in ef for t in f]
                eu = graph_compose(table, union, e.source, e.target, vocab)
                preds_ext = {"ext": ex_res[0][0] or free_by[i][0], "ext_only": ex_res[0][0],
                             "ext_sc": _vote([r for r, _ in ex_res[1:]]) or sc_vote,
                             "ext_union": eu[0] or free_by[i][0],
                             "ext_all": _vote([r for r, _ in ex_res]) or _vote([r for r, _ in gsym]) or sc_vote,
                             "ext_lex": lex_res[0][0] or free_by[i][0],
                             "ext_g": g_res[0][0] or free_by[i][0],
                             "ext_glex": gl_res[0][0] or free_by[i][0],
                             "ext_g_sc": _vote([r for r, _ in g_res[1:]]) or sc_vote,
                             "ext_glex_sc": _vote([r for r, _ in gl_res[1:]]) or sc_vote,
                             "ext_glex_all": _vote([r for r, _ in gl_res]) or sc_vote}
                if args.ext_llm:
                    preds_ext["ext_llm"] = llm_ext[i]
            preds = {
                "graph": gsym[0][0] or free_by[i][0],
                "graph_sc": _vote([r for r, _ in gsym[1:]]) or sc_vote,
                **preds_ext,
                "cot_free": free_by[i][0],
                "cot_score": cot_score,
                "sc": _vote([free_by[i][j] for j in range(1, len(chains))]) if samples[i] else None,
                "sc_plus": _vote([free_by[i][j] for j in range(1, len(chains) + len(extra_s[i]))])
                if samples[i] else None,
                "sym": sym[0][0] or free_by[i][0],
                "sym_only": sym[0][0],
                "sym_sc": _vote([r for r, _ in sym[1:]]) or (
                    _vote([free_by[i][j] for j in range(1, len(chains))]) if samples[i] else free_by[i][0]),
                "sym_gold": gold_sym,
            }
            for mth, p in preds.items():
                rows.append(dict(uid=e.uid, k=e.k, method=mth, pred=p, gold=e.answer,
                                 correct=float(p == e.answer), sym_status=sym[0][1],
                                 n_valid_sym=sum(r is not None for r, _ in sym), hop_fidelity=fid,
                                 graph_status=gsym[0][1],
                                 ext_status=(graph_compose(table, parse_facts(ext_g[i]), e.source, e.target,
                                                           vocab)[1] if args.extract else None),
                                 chain=greedy[i] if mth == "cot_free" else None,
                                 extract=ext_g[i] if mth == "cot_free" else None))
        _log(f"  {min(s + args.chunk, len(ex))}/{len(ex)} done "
             f"({(time.time() - T0) / 60:.0f} min)")
        pd.DataFrame(rows).to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
    return pd.DataFrame(rows)


# ============================================================================ MuSiQue: evidence selection

def _ctx_text(paras: List[dict]) -> str:
    return "\n".join(f"[{i + 1}] {p['title']}: {p['paragraph_text']}" for i, p in enumerate(paras))


def _set_attn(model, impl: str) -> str:
    old = getattr(model.config, "_attn_implementation", "sdpa")
    for m in [model] + [x for x in model.modules() if hasattr(x, "config")]:
        try:
            m.config._attn_implementation = impl
        except Exception:
            pass
    return old


@torch.no_grad()
def attention_scores(lm, paras: List[dict], question: str) -> np.ndarray:
    """Attention mass from the question tokens to each paragraph, summed over layers and heads,
    minus the same quantity for a content-free question (calibration as in ICR, ICLR 2025)."""
    ctx = _ctx_text(paras)
    full = lm.prompt(musique_messages(ctx, question))
    null = lm.prompt(musique_messages(ctx, "N/A"))
    cut = full.rfind("\nQuestion: ")
    prefix = full[:cut]
    assert null.startswith(prefix)
    enc = lm.tok(prefix, return_offsets_mapping=True, add_special_tokens=False)
    offs = np.array([o[0] for o in enc["offset_mapping"]])
    base = prefix.rfind("Paragraphs:\n")
    spans, pos = [], base
    for i, p in enumerate(paras):
        line = f"[{i + 1}] {p['title']}: {p['paragraph_text']}"
        a = prefix.index(line, pos)
        spans.append((a, a + len(line)))
        pos = a + len(line)
    tok_spans = [np.where((offs >= a) & (offs < b))[0] for a, b in spans]
    ids = torch.tensor([enc["input_ids"]], device=lm.device)
    suffixes = [lm.tok(t, return_tensors="pt", add_special_tokens=False).input_ids.to(lm.device)
                for t in (full[cut:], null[cut:])]
    res = _attn_pass(lm, ids, suffixes, tok_spans)

    def _bad(r):   # non-finite, or degenerate (all paragraphs scored alike)
        d = r[0] - r[1]
        return (not all(np.isfinite(x).all() for x in r)) or (len(d) > 1 and np.ptp(d) == 0)
    if _bad(res):
        # fp16 eager attention overflows on some models (seen on Qwen2.5-1.5B): redo in fp32
        if getattr(lm.model, "hf_device_map", None) is None:
            dt = next(lm.model.parameters()).dtype
            lm.model.float()
            try:
                res = _attn_pass(lm, ids, suffixes, tok_spans)
            finally:
                lm.model.to(dt)
            ATTN_STATS["fp32_retry"] += 1
        if _bad(res):
            ATTN_STATS["failed"] += 1
            raise RuntimeError("non-finite attention scores")
    ATTN_STATS["ok"] += 1
    return res[0] - res[1]


ATTN_STATS: Counter = Counter()


def _attn_pass(lm, ids, suffixes, tok_spans) -> List[np.ndarray]:
    past = lm.model(input_ids=ids, use_cache=True).past_key_values
    L = ids.shape[1]
    res = []
    old = _set_attn(lm.model, "eager")
    try:
        for sid in suffixes:
            out = lm.model(input_ids=sid, past_key_values=copy.deepcopy(past), use_cache=True,
                           output_attentions=True,
                           cache_position=torch.arange(L, L + sid.shape[1], device=lm.device))
            if out.attentions is None or out.attentions[0] is None:
                raise RuntimeError("model returned no attentions")
            a = sum(x[0, :, :, :L].float().sum(0).mean(0).cpu() for x in out.attentions).numpy()
            res.append(np.array([a[t].sum() if len(t) else 0.0 for t in tok_spans]))
    finally:
        _set_attn(lm.model, old)
    return res


SELECT_SYSTEM = ("You are given numbered paragraphs and a multi-hop question. Do not answer the "
                 "question. List the numbers of all paragraphs needed to answer it, in the form "
                 "'Relevant: 2, 5'.")


def _chain_and_answer(lm, contexts: List[str], questions: List[str], bs: int, sample_n: int = 0):
    prompts = [lm.prompt(musique_messages(c, q)) for c, q in zip(contexts, questions)]
    if sample_n:
        chains = lm.generate(prompts, max_new_tokens=200, do_sample=True, n=sample_n,
                             batch_size=max(1, bs // 2))
        prompts = [p for p in prompts for _ in range(sample_n)]
    else:
        chains = lm.generate(prompts, max_new_tokens=200, batch_size=bs)
    p2 = [p + C.render_chain(C.parse_chain(g)) + "Answer:" for p, g in zip(prompts, chains)]
    ans = [a.strip().strip(".") for a in lm.generate(p2, max_new_tokens=24, batch_size=bs, stop="\n")]
    return chains, ans


def run_musique(lm, args) -> pd.DataFrame:
    ms = D.load_musique("dev")
    rng = np.random.default_rng(args.seed + 3)          # same draw as the mediation runs
    ex = [ms[i] for i in rng.choice(len(ms), args.n, replace=False)]
    _log(f"MuSiQue: {len(ex)} examples x {args.conditions}")
    rows = []
    for cond in args.conditions:
        exs = [D.musique_condition(e, cond) for e in ex]
        for s in range(0, len(exs), args.chunk):
            if (time.time() - T0) / 60 > args.max_minutes:
                _log("time budget reached; stopping early")
                break
            part = exs[s:s + args.chunk]
            paras = [e.context() for e in part]
            qs = [e.question for e in part]
            golds = [[e.answer] + list(e.aliases) for e in part]
            preds: Dict[str, List] = {}
            extra: Dict[str, List] = defaultdict(lambda: [None] * len(part))
            # baseline: greedy CoT + free answer; last-hop readout from the same chain
            ch, an = _chain_and_answer(lm, [_ctx_text(p) for p in paras], qs, args.bs)
            preds["cot_free"] = an
            preds["last_hop"] = [(C.parse_chain(c)[-1].result if C.parse_chain(c) else a)
                                 for c, a in zip(ch, an)]
            # self-consistency over sampled chains
            if args.sc_n:
                _, sa = _chain_and_answer(lm, [_ctx_text(p) for p in paras], qs, args.bs, args.sc_n)
                preds["sc"] = [_vote(sa[i * args.sc_n:(i + 1) * args.sc_n], M._norm_ans)
                               for i in range(len(part))]
            M_ = set(args.ms_methods)
            # attention scores (computed once, reused by every attention variant)
            scores = [None] * len(part)
            if M_ & {"attn_sel", "attn_rerank", "attn_drop2"}:
                for i, p in enumerate(paras):
                    try:
                        scores[i] = attention_scores(lm, p, qs[i])
                    except Exception as err:          # never lose the run to one example
                        _log(f"attention scoring failed ({err}); keeping all paragraphs")
                        scores[i] = np.zeros(len(p))

            def _attn_ctx(name, pick):
                ctxs = []
                for i, p in enumerate(paras):
                    keep = pick(scores[i], len(p))
                    kept = [p[j] for j in keep]
                    sup = [q for q in p if q["is_supporting"]]
                    extra[name + "_recall"][i] = sum(q["is_supporting"] for q in kept) / max(1, len(sup))
                    ctxs.append(_ctx_text(kept))
                return ctxs
            if "attn_sel" in M_:     # hard filter: top-K paragraphs, original order
                c = _attn_ctx("attn", lambda sc, n: sorted(np.argsort(-sc)[:args.top_k].tolist()))
                _, preds["attn_sel"] = _chain_and_answer(lm, c, qs, args.bs)
            if "attn_rerank" in M_:  # soft: keep every paragraph, most-attended first
                c = _attn_ctx("rerank", lambda sc, n: np.argsort(-sc).tolist())
                _, preds["attn_rerank"] = _chain_and_answer(lm, c, qs, args.bs)
            if "attn_drop2" in M_:   # mild filter: drop only the two least-attended paragraphs
                c = _attn_ctx("drop2", lambda sc, n: sorted(np.argsort(-sc)[:max(1, n - 2)].tolist()))
                _, preds["attn_drop2"] = _chain_and_answer(lm, c, qs, args.bs)
            if "self_sel" in M_:     # self-selection by prompting, then CoT
                sp = [lm.prompt([{"role": "system", "content": SELECT_SYSTEM},
                                 {"role": "user", "content": f"Paragraphs:\n{_ctx_text(p)}\nQuestion: {q}"}],
                                assistant_prefix="Relevant:") for p, q in zip(paras, qs)]
                raw = lm.generate(sp, max_new_tokens=24, batch_size=args.bs, stop="\n")
                self_ctx = []
                for i, (p, r) in enumerate(zip(paras, raw)):
                    nums = sorted({int(x) - 1 for x in re.findall(r"\d+", r) if 0 < int(x) <= len(p)})
                    kept = [p[j] for j in nums] if nums else p
                    sup = [q for q in p if q["is_supporting"]]
                    extra["self_recall"][i] = sum(q["is_supporting"] for q in kept) / max(1, len(sup))
                    extra["self_kept"][i] = len(kept)
                    self_ctx.append(_ctx_text(kept))
                _, preds["self_sel"] = _chain_and_answer(lm, self_ctx, qs, args.bs)
            if "gold_only" in M_:    # oracle ceiling: supporting paragraphs only
                _, preds["gold_only"] = _chain_and_answer(
                    lm, [_ctx_text([q for q in p if q["is_supporting"]]) for p in paras], qs, args.bs)
            for i, e in enumerate(part):
                for mth, pl in preds.items():
                    p = pl[i] or ""
                    rows.append(dict(uid=e.uid, k=e.k, cond=cond, method=mth, pred=p,
                                     em=M.em(p, golds[i]), f1=M.f1(p, golds[i]),
                                     attn_recall=extra["attn_recall"][i],
                                     drop2_recall=extra["drop2_recall"][i],
                                     self_recall=extra["self_recall"][i],
                                     self_kept=extra["self_kept"][i], n_paras=len(paras[i]),
                                     chain=ch[i] if mth == "cot_free" else None))
            _log(f"  [{cond}] {min(s + args.chunk, len(exs))}/{len(exs)} "
                 f"({(time.time() - T0) / 60:.0f} min)")
            pd.DataFrame(rows).to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
    return pd.DataFrame(rows)


# ============================================================================ analysis

def summarize(df: pd.DataFrame, metric: str, by: List[str], base: List[str]) -> pd.DataFrame:
    """Mean with bootstrap CI per method, and paired difference + McNemar against each baseline."""
    out = []
    for key, g in df.groupby(by):
        key = key if isinstance(key, tuple) else (key,)
        w = g.pivot_table(index="uid", columns="method", values=metric)
        for mth in w.columns:
            mu, lo, hi = M.boot_ci(w[mth].values)
            rec = dict(zip(by, key), method=mth, mean=mu, lo=lo, hi=hi, n=int(w[mth].notna().sum()))
            for b in base:
                if b in w.columns and b != mth:
                    d = (w[mth] - w[b]).dropna().values
                    dm, dlo, dhi = M.boot_ci(d)
                    rec[f"d_vs_{b}"], rec[f"d_lo_{b}"], rec[f"d_hi_{b}"] = dm, dlo, dhi
                    if metric in ("correct", "em"):
                        rec[f"p_vs_{b}"] = M.mcnemar_p(w[mth].values >= 0.5, w[b].values >= 0.5)
            out.append(rec)
    return pd.DataFrame(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=["clutrr", "musique"], required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--device_map", default=None)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--sc_n", type=int, default=5)
    ap.add_argument("--sc_extra", type=int, default=0,
                    help="extra sampled chains for a compute-matched self-consistency baseline (sc_plus)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--chunk", type=int, default=40)
    ap.add_argument("--max_minutes", type=float, default=600)
    ap.add_argument("--ks", type=int, nargs="+", default=[2, 3, 4, 6, 8])
    ap.add_argument("--n_per_k", type=int, default=60)
    ap.add_argument("--n", type=int, default=300)
    ap.add_argument("--conditions", nargs="+", default=["orig", "dist8"])
    ap.add_argument("--top_k", type=int, default=4)
    ap.add_argument("--extract", type=int, default=1)
    ap.add_argument("--extract_genders", type=int, default=1,
                    help="ask the extractor to also state each person's gender (from pronouns)")
    ap.add_argument("--split", default="test", choices=["test", "validation"])
    ap.add_argument("--ext_llm", type=int, default=0,
                    help="ablation: LLM answers from its own extracted facts (no rule table)")
    ap.add_argument("--task_name", default=D.CLUTRR_TASK,
                    help="CLUTRR task folder to draw evaluation stories from (rules always come from "
                         "the gen_train234 training split)")
    ap.add_argument("--ms_methods", nargs="+",
                    default=["attn_sel", "self_sel", "gold_only"],
                    help="MuSiQue methods besides cot_free/last_hop/sc: attn_sel attn_rerank attn_drop2 "
                         "self_sel gold_only")
    args = ap.parse_args()
    os.makedirs(args.outdir, exist_ok=True)
    json.dump(vars(args), open(os.path.join(args.outdir, "config.json"), "w"), indent=1)
    from .lm import LM
    lm = LM(args.model, device_map=args.device_map)
    _log(f"loaded {args.model} on {lm.device}")
    if args.task == "clutrr":
        df = run_clutrr(lm, args)
        summ = summarize(df[df.method != "sym_gold"].assign(cond="all"), "correct", ["cond"],
                         ["cot_free", "sc", "sc_plus", "cot_score"])
        g = df[df.method == "cot_free"]
        byk = summarize(df.assign(cond="all"), "correct", ["k"], ["cot_free"])
        byk.to_csv(os.path.join(args.outdir, "by_k.csv"), index=False)
        diag = g.sym_status.value_counts(normalize=True).to_dict()
        extra = dict(sym_status=diag, graph_status=g.graph_status.value_counts(normalize=True).to_dict(),
                     ext_status=g.ext_status.value_counts(normalize=True).to_dict(), hop_fidelity=float(df[df.method == "cot_free"].hop_fidelity.mean()),
                     sym_gold_acc=float(df[df.method == "sym_gold"].correct.mean()))
    else:
        df = run_musique(lm, args)
        summ = pd.concat([summarize(df, "em", ["cond"], ["cot_free", "sc"]).assign(metric="em"),
                          summarize(df, "f1", ["cond"], ["cot_free", "sc"]).assign(metric="f1")])
        g = df[df.method == "cot_free"]
        extra = dict(attn_stats=dict(ATTN_STATS),
                     attn_recall=float(g.attn_recall.mean()), drop2_recall=float(g.drop2_recall.mean()),
                     self_recall=float(g.self_recall.mean()),
                     self_kept=float(g.self_kept.mean()), n_paras=float(g.n_paras.mean()))
    df.to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
    summ.to_csv(os.path.join(args.outdir, "summary.csv"), index=False)
    extra["minutes"] = (time.time() - T0) / 60
    json.dump(dict(model=args.model, task=args.task, **extra), open(os.path.join(args.outdir, "summary.json"), "w"),
              indent=1, default=float)
    with pd.option_context("display.width", 250, "display.max_columns", 30):
        _log("\n" + summ.round(3).to_string())
    _log(f"extra: {extra}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hgcd/cftrain.py
"""Round 10: train reading-friendly chains on CLUTRR and evaluate them.

Diagnosis (R3, D3): models copy what is written in a chain but do not compose k relations in one
step. Here every hop is followed by a *running relation* line ("So Y is <source>'s <c_i>."), so each
step is a one-step lookup and the final answer is a copy of the last line. A LoRA model is trained on
k in {2,3,4} only and evaluated on the round-9 sets (same sampling, so baselines join on uid), on
longer chains (k=5-8) and under surface perturbations. No rule table is used at inference.

    python -m hgcd.cftrain --model Qwen/Qwen2.5-3B-Instruct --outdir out --arms p1 s_plain s_struct
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import os
import random
import re
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from . import chain as C
from . import data as D
from . import metrics as M
from .causaldec import _first_rel, induce_table, parse_triples
from .prompts import CLUTRR_SHOTS, CLUTRR_SYSTEM, _clutrr_user
from .lm import LM

T0 = time.time()

STRUCT_SYSTEM = (
    "You solve family-relation puzzles. Walk from the first person to the second person one "
    "relation at a time, using only facts stated in the story. For every hop write "
    "'Hop i: X's <relation> is Y. => Y', then on the next line the running relation "
    "'So Y is <first person>'s <relation>.' obtained by combining the previous running relation "
    "with the new hop. End with the final answer.")

# running relations for the two few-shot examples of prompts.CLUTRR_SHOTS
_SHOT_RUN = [["son", "daughter"], ["father", "aunt", "cousin"]]

SETS = {  # name -> (task, split, ks, n_per_k, seed)  identical to rounds 9 (C4, C6, C5)
    "c4": (D.CLUTRR_TASK, "validation", [2, 3, 4], 100, 2),
    "c6": ("gen_train23_test2to10", "test", [2, 3, 4], 60, 7),
    "c5": ("gen_train23_test2to10", "test", [5, 6, 7, 8], 50, 5),
}
ROBUST_SET = "c6"
ROBUST_CONDS = ["rename", "order", "distract"]
_SO = re.compile(r"^\s*So\s+(.+?)\s+is\s+(.+?)['’]s\s+([A-Za-z\- ]+?)\s*\.?\s*$")


def _log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ----------------------------------------------------------------------------- chains / prompts

def chain_text(ex: D.ClutrrExample, table: Dict, structured: bool) -> Optional[str]:
    """Gold chain (plain) or gold chain with a running relation after each hop (structured).
    None if the stepwise composition is undefined or disagrees with the gold answer."""
    lines, c = [], None
    for i, h in enumerate(ex.gold_hops):
        lines.append(f"Hop {i + 1}: {h.text()} => {h.tail}")
        if structured:
            c = h.rel if i == 0 else table.get((c, h.rel))
            if c is None:
                return None
            lines.append(f"So {h.tail} is {ex.source}'s {c}.")
    if structured and c != ex.answer:
        return None
    return "\n".join(lines) + f"\nAnswer: {ex.target} is {ex.source}'s {ex.answer}."


def extend_table(train: List[D.ClutrrExample], table: Dict, vocab: List[str], iters: int = 10,
                 purity: float = 0.9, min_count: int = 3) -> Dict:
    """Abductive extension of the pairwise table (train gold only): for a chain whose running relation
    is undefined at exactly one step, the answer pins down the missing relation if one candidate
    reproduces it."""
    table = dict(table)

    def first_missing(e):
        c = None
        for i, h in enumerate(e.gold_hops):
            n = h.rel if c is None else table.get((c, h.rel))
            if n is None:
                return i, c
            c = n
        return None, c

    def finish(e, i, x):
        c = x
        for h in e.gold_hops[i + 1:]:
            c = table.get((c, h.rel))
            if c is None:
                return None
        return c

    for _ in range(iters):
        obs: Dict[Tuple[str, str], Dict[str, int]] = {}
        for e in train:
            i, c = first_missing(e)
            if i is None or i == 0:
                continue
            ok = [x for x in vocab if finish(e, i, x) == e.answer]
            if len(ok) == 1:
                d = obs.setdefault((c, e.gold_hops[i].rel), {})
                d[ok[0]] = d.get(ok[0], 0) + 1
        new = {k: max(v, key=v.get) for k, v in obs.items() if k not in table and sum(v.values()) >= min_count
               and max(v.values()) >= purity * sum(v.values())}
        if not new:
            break
        table.update(new)
    return table


def covered(ex: D.ClutrrExample, table: Dict) -> bool:
    run = gold_running(ex, table)
    return all(r is not None for r in run) and run[-1] == ex.answer


def gold_running(ex: D.ClutrrExample, table: Dict) -> List[Optional[str]]:
    out, c = [], None
    for i, h in enumerate(ex.gold_hops):
        c = h.rel if i == 0 else (table.get((c, h.rel)) if c is not None else None)
        out.append(c)
    return out


def msgs_zero(system: str, ex: D.ClutrrExample) -> List[dict]:
    return [{"role": "system", "content": system},
            {"role": "user", "content": _clutrr_user(ex.story_text, ex.source, ex.target)}]


def msgs_struct_fewshot(ex: D.ClutrrExample) -> List[dict]:
    m = [{"role": "system", "content": STRUCT_SYSTEM}]
    for s, runs in zip(CLUTRR_SHOTS, _SHOT_RUN):
        lines = []
        for i, ((body, res), c) in enumerate(zip(s["hops"], runs)):
            lines.append(f"Hop {i + 1}: {body} => {res}")
            lines.append(f"So {res} is {s['src']}'s {c}.")
        m.append({"role": "user", "content": _clutrr_user(s["story"], s["src"], s["tgt"])})
        m.append({"role": "assistant", "content": "\n".join(lines) +
                  f"\nAnswer: {s['tgt']} is {s['src']}'s {s['ans']}."})
    m.append({"role": "user", "content": _clutrr_user(ex.story_text, ex.source, ex.target)})
    return m


# ----------------------------------------------------------------------------- training

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def _tokenize(tok, prompt: str, target: str, max_len: int = 2048):
    p = tok(prompt, add_special_tokens=False).input_ids
    t = tok(target + tok.eos_token, add_special_tokens=False).input_ids
    ids = (p + t)[:max_len]
    labels = ([-100] * len(p) + t)[:max_len]
    return ids, labels


def train_sft(lm: LM, data: List[Tuple[str, str]], args) -> Dict:
    """LoRA SFT on (prompt, target) pairs; loss on the target only. Merges the adapter into lm.model."""
    from peft import LoraConfig, get_peft_model
    tok, model = lm.tok, lm.model
    cuda = torch.cuda.is_available()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                          target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    params = [p for p in pm.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0)
    enc = [_tokenize(tok, p, t) for p, t in data]
    rng = random.Random(args.seed)
    n_steps = math.ceil(len(enc) * args.epochs / (args.mb * args.accum))
    warm = max(1, int(0.05 * n_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / n_steps))))
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    pm.train()
    dev = lm.device
    losses, step, micro = [], 0, 0
    order: List[int] = []
    for _ in range(math.ceil(args.epochs)):
        o = list(range(len(enc)))
        rng.shuffle(o)
        order += o
    order = order[:int(len(enc) * args.epochs)]
    t_start = time.time()
    for b in range(0, len(order), args.mb):
        batch = [enc[i] for i in order[b:b + args.mb]]
        L = max(len(x[0]) for x in batch)
        ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for i, (x, y) in enumerate(batch):
            ids[i, :len(x)] = torch.tensor(x)
            lab[i, :len(y)] = torch.tensor(y)
            att[i, :len(x)] = 1
        ids, lab, att = ids.to(dev), lab.to(dev), att.to(dev)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            logits = pm(input_ids=ids, attention_mask=att).logits
        loss = torch.nn.functional.cross_entropy(
            logits[:, :-1].float().reshape(-1, logits.size(-1)), lab[:, 1:].reshape(-1),
            ignore_index=-100)
        scaler.scale(loss / args.accum).backward()
        losses.append(loss.item())
        micro += 1
        if micro % args.accum == 0 or b + args.mb >= len(order):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            if step % 10 == 0 or step == n_steps:
                _log(f"  sft step {step}/{n_steps} loss {np.mean(losses[-10 * args.accum:]):.4f} "
                     f"({(time.time() - t_start) / 60:.1f} min)")
    pm.eval()
    merged = pm.merge_and_unload()
    merged.gradient_checkpointing_disable()
    merged.config.use_cache = True
    merged.eval()
    lm.model = merged
    return dict(steps=step, final_loss=float(np.mean(losses[-20:])), n_train=len(enc),
                minutes=(time.time() - t_start) / 60)


# ----------------------------------------------------------------------------- evaluation

def _step_acc(chain: str, ex: D.ClutrrExample, table: Dict) -> Tuple[float, float]:
    """(fraction of running-relation lines that are right, 1 if the last one is right)."""
    gold = gold_running(ex, table)
    got = [m.group(3).strip().lower() for ln in chain.splitlines() if (m := _SO.match(ln))]
    if not got:
        return float("nan"), float("nan")
    ok = [i < len(gold) and g == gold[i] for i, g in enumerate(got)]
    return float(np.mean(ok)), float(len(got) == len(gold) and got[-1] == ex.answer)


def evaluate(lm: LM, arm: str, msg_fn, args, table: Dict, vocab_l: List[str], sets, robust: bool,
             max_new_fn) -> List[Dict]:
    rows = []
    for name in sets:
        task, split, ks, npk, seed = SETS[name]
        npk = max(1, int(round(npk * args.eval_scale)))
        pool = D.load_clutrr(split, task=task)
        base = D.sample_by_k(pool, ks, npk, seed)
        conds = ["orig"] + (ROBUST_CONDS if (robust and name == ROBUST_SET) else [])
        for cond in conds:
            ex = [D.apply_condition(e, cond, pool, seed) for e in base]
            prompts = [lm.prompt(msg_fn(e)) for e in ex]
            chains: List[str] = [""] * len(ex)
            for k in sorted(set(e.k for e in ex)):
                idx = [i for i, e in enumerate(ex) if e.k == k]
                if max_new_fn(k) <= 0:       # answer-only models write no chain
                    continue
                out = lm.generate([prompts[i] for i in idx], max_new_tokens=max_new_fn(k),
                                  batch_size=args.bs)
                for t, i in enumerate(idx):
                    chains[i] = out[t].rstrip() + "\n"
            ans = lm.generate([p + c + e.answer_prefix() for p, c, e in zip(prompts, chains, ex)],
                              max_new_tokens=8, batch_size=args.bs * 2, stop="\n")
            for e, c, a in zip(ex, chains, ans):
                trip = parse_triples(c)
                gold_t = {(C.norm(h.head), h.rel, C.norm(h.tail)) for h in e.gold_hops}
                fid = float(np.mean([(C.norm(x), r, C.norm(y)) in gold_t for x, r, y in trip])) if trip else 0.0
                sa, last_ok = _step_acc(c, e, table)
                pred = _first_rel(a, vocab_l)
                rows.append(dict(arm=arm, set=name, cond=cond, uid=e.uid, k=e.k, pred=pred, gold=e.answer,
                                 correct=float(pred == e.answer), hop_fidelity=fid, step_acc=sa,
                                 last_step_ok=last_ok, covered=int(covered(e, table)), chain=c))
            acc = np.mean([r["correct"] for r in rows if r["set"] == name and r["cond"] == cond
                           and r["arm"] == arm])
            _log(f"  [{arm}] {name}/{cond}: n={len(ex)} acc={100 * acc:.1f}")
    return rows


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (arm, st, cond), g in df.groupby(["arm", "set", "cond"]):
        m, lo, hi = M.boot_ci(g.correct.values)
        row = dict(arm=arm, set=st, cond=cond, n=len(g), acc=100 * m, lo=100 * lo, hi=100 * hi,
                   hop_fidelity=g.hop_fidelity.mean(), step_acc=g.step_acc.mean(),
                   last_step_ok=g.last_step_ok.mean())
        if cond != "orig":  # flip rate against the same model's unperturbed answer
            o = df[(df.arm == arm) & (df.set == st) & (df.cond == "orig")].set_index("uid").pred
            p = g.set_index("uid").pred
            common = p.index.intersection(o.index)
            row["flip_rate"] = float((p[common] != o[common]).mean())
        out.append(row)
    return pd.DataFrame(out)


# ----------------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--arms", nargs="+", default=["p1", "s_plain", "s_struct"])
    ap.add_argument("--sets", nargs="+", default=["c4", "c6", "c5"])
    ap.add_argument("--n_train", type=int, default=3000)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--mb", type=int, default=2)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--eval_scale", type=float, default=1.0)
    ap.add_argument("--robust", type=int, default=1)
    ap.add_argument("--device_map", default=None)
    ap.add_argument("--max_minutes", type=float, default=600.0)
    args = ap.parse_args()
    os.makedirs(args.outdir, exist_ok=True)
    json.dump(vars(args), open(os.path.join(args.outdir, "config.json"), "w"), indent=1)

    train = D.load_clutrr("train")
    vocab_l = D.clutrr_relations(train)
    table = extend_table(train, induce_table(train), vocab_l)
    pool = [e for e in train if 2 <= e.k <= 4]
    sample = D.sample_by_k(pool, [2, 3, 4], args.n_train // 3, args.seed)
    _log(f"train pool {len(train)} stories, {len(table)} rules; sampled {len(sample)} (k=2-4)")

    all_rows: List[Dict] = []
    info: Dict[str, Dict] = {}
    for arm in args.arms:
        if (time.time() - T0) / 60 > args.max_minutes:
            _log(f"[{arm}] SKIPPED: past max_minutes")
            continue
        _log(f"=== arm {arm}")
        lm = LM(args.model, device_map=args.device_map)
        if arm == "p1":
            msg_fn, mx = msgs_struct_fewshot, lambda k: 44 * (k + 1) + 16
        else:
            structured = arm == "s_struct"
            data, dropped = [], 0
            for e in sample:
                t = chain_text(e, table, structured)
                if t is None:
                    dropped += 1
                    continue
                data.append((lm.prompt(msgs_zero(STRUCT_SYSTEM if structured else CLUTRR_SYSTEM, e)), t))
            _log(f"  {len(data)} training chains ({dropped} dropped: stepwise rule undefined/inconsistent)")
            info[arm] = train_sft(lm, data, args)
            info[arm]["dropped"] = dropped
            system = STRUCT_SYSTEM if structured else CLUTRR_SYSTEM
            msg_fn = lambda e, s=system: msgs_zero(s, e)
            mx = (lambda k: 44 * (k + 1) + 16) if structured else (lambda k: 26 * (k + 2) + 16)
        all_rows += evaluate(lm, arm, msg_fn, args, table, vocab_l, args.sets, bool(args.robust), mx)
        df = pd.DataFrame(all_rows)
        df.to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
        summarize(df).round(3).to_csv(os.path.join(args.outdir, "summary.csv"), index=False)
        json.dump(info, open(os.path.join(args.outdir, "summary.json"), "w"), indent=1)
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    if all_rows:
        print(summarize(pd.DataFrame(all_rows)).round(3).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hgcd/egen.py
"""Round 10b: does the structured-chain recipe generalise beyond CLUTRR? (MuSiQue, 2WikiMultihopQA)

Recipe: every step is a one-step lookup whose result is written down and read by the next step; the
model is LoRA-trained on such chains built automatically from annotated decompositions, and compared
with (i) answer-only SFT, (ii) SFT on plain hop chains, (iii) the base model with a few-shot chain.

    python -m hgcd.egen --dataset musique --model Qwen/Qwen2.5-3B-Instruct --outdir out \
        --arms p0 s_ans s_plain s_struct
"""

from __future__ import annotations

import argparse
import gc
import json
import os
import random
import re
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch

from . import data as D
from . import metrics as M
from .cftrain import train_sft
from .lm import LM
from .prompts import musique_messages

T0 = time.time()

SYS_ANS = ("You answer multi-hop questions using the given paragraphs. Reply with the final answer "
           "only, in the form 'Answer: <answer>'.")
SYS_PLAIN = ("You answer multi-hop questions using the given paragraphs. Break the question into "
             "single-hop sub-questions, answer each in one line of the form "
             "'Hop i: <sub-question> => <answer>', then give the final answer as 'Answer: <answer>'.")
SYS_STRUCT = ("You answer multi-hop questions using the given paragraphs. Break the question into "
              "single-hop sub-questions. For each hop write 'Hop i: <sub-question> => <answer> "
              "[from: <title of the paragraph that states it>]'. The next sub-question must use the "
              "answer of the previous hop. Then give the final answer as 'Answer: <answer>'.")
_FROM = re.compile(r"\[from:\s*(.+?)\s*\]\s*$")


def _log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


@dataclass
class QA:
    uid: str
    question: str
    answer: str
    aliases: List[str]
    context: str                      # "[i] Title: text" lines
    titles: List[str]                 # titles shown in the context
    hops: List[tuple]                 # (hop text, result, source title or None)
    support: List[str]                # gold supporting titles
    kind: str = ""
    cond: str = "orig"

    @property
    def k(self) -> int:
        return len(self.hops)


def _norm_title(t: str) -> str:
    return re.sub(r"\s*\(.*?\)\s*", " ", t.lower()).strip()


def _find_title(name: str, titles: List[str]) -> Optional[str]:
    n = _norm_title(name)
    for t in titles:
        if _norm_title(t) == n:
            return t
    for t in titles:
        tn = _norm_title(t)
        if n and (n in tn or tn in n) and len(tn) > 2:
            return t
    return None


# ----------------------------------------------------------------------------- loaders

def load_musique_qa(split: str, n_dist: int = 4) -> List[QA]:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("bdsaglam/musique", f"musique_ans_v1.0_{split}.jsonl", repo_type="dataset")
    out = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            paras = d["paragraphs"]
            sup = [x for x in paras if x["is_supporting"]]
            dis = [x for x in paras if not x["is_supporting"]]
            rng = random.Random(f"ms-dist-{d['id']}")
            rng.shuffle(dis)
            shown = sup + dis[:n_dist]
            random.Random(f"ms-order-{d['id']}-0").shuffle(shown)
            steps = d["question_decomposition"]
            answers = [s["answer"] for s in steps]
            hops = []
            for i, s in enumerate(steps):
                idx = s.get("paragraph_support_idx")
                title = paras[idx]["title"] if isinstance(idx, int) and 0 <= idx < len(paras) else None
                hops.append((D._render_subq(s["question"], answers[:i]), s["answer"], title))
            out.append(QA(uid=d["id"], question=d["question"], answer=d["answer"],
                          aliases=d.get("answer_aliases") or [],
                          context="\n".join(f"[{i + 1}] {x['title']}: {x['paragraph_text']}"
                                            for i, x in enumerate(shown)),
                          titles=[x["title"] for x in shown], hops=hops,
                          support=[x["title"] for x in sup], kind=f"{len(steps)}hop"))
    return out


def load_wiki2_qa(split: str, types=("compositional", "inference")) -> List[QA]:
    from huggingface_hub import hf_hub_download
    files = {"train": ["data/train-00000-of-00002.parquet", "data/train-00001-of-00002.parquet"],
             "validation": ["data/validation-00000-of-00001.parquet"]}[split]
    df = pd.concat([pd.read_parquet(hf_hub_download("framolfese/2WikiMultihopQA", f, repo_type="dataset"))
                    for f in files])
    out = []
    for r in df.itertuples():
        if r.type not in types or len(r.evidences) < 2:
            continue
        titles = list(r.context["title"])
        sents = [" ".join(s) for s in r.context["sentences"]]
        hops = []
        for ev in r.evidences:
            head, rel, tail = (str(x) for x in ev)
            hops.append((f"What is the {rel} of {head}?", tail, _find_title(head, titles)))
        out.append(QA(uid=str(r.id), question=r.question, answer=str(r.answer), aliases=[],
                      context="\n".join(f"[{i + 1}] {t}: {s}" for i, (t, s) in enumerate(zip(titles, sents))),
                      titles=titles, hops=hops, support=list(dict.fromkeys(r.supporting_facts["title"])),
                      kind=r.type))
    return out


# ----------------------------------------------------------------------------- prompts / targets

def user_msg(e: QA) -> str:
    return f"Paragraphs:\n{e.context}\nQuestion: {e.question}"


def msgs_zero(system: str, e: QA) -> List[dict]:
    return [{"role": "system", "content": system}, {"role": "user", "content": user_msg(e)}]


def target(e: QA, arm: str) -> Optional[str]:
    if arm == "s_ans":
        return f"Answer: {e.answer}"
    lines = []
    for i, (q, a, t) in enumerate(e.hops):
        if arm == "s_struct":
            if t is None:
                return None            # every step must cite the paragraph it reads
            lines.append(f"Hop {i + 1}: {q} => {a} [from: {t}]")
        else:
            lines.append(f"Hop {i + 1}: {q} => {a}")
    return "\n".join(lines) + f"\nAnswer: {e.answer}"


# ----------------------------------------------------------------------------- evaluation

def evaluate(lm: LM, arm: str, dataset: str, exs: Dict[str, List[QA]], args) -> List[Dict]:
    rows = []
    system = {"s_ans": SYS_ANS, "s_plain": SYS_PLAIN, "s_struct": SYS_STRUCT}.get(arm)
    for cond, items in exs.items():
        prompts = [lm.prompt(musique_messages(e.context, e.question) if arm == "p0"
                             else msgs_zero(system, e)) for e in items]
        if arm == "s_ans":
            chains = [""] * len(items)
        else:
            chains = []
            for s in range(0, len(items), args.bs):
                chains += lm.generate(prompts[s:s + args.bs], max_new_tokens=60 * 4 + 40,
                                      batch_size=args.bs)
        chains = [c.rstrip() + "\n" if c.strip() else "" for c in chains]
        ans = lm.generate([p + c + "Answer:" for p, c in zip(prompts, chains)], max_new_tokens=24,
                          batch_size=args.bs, stop="\n")
        for e, c, a in zip(items, chains, ans):
            pred = a.strip().strip(".")
            golds = [e.answer] + list(e.aliases)
            lines = [ln for ln in c.splitlines() if ln.startswith("Hop")]
            cites = [m.group(1) for ln in lines if (m := _FROM.search(ln))]
            last = lines[-1].split("=>")[-1] if lines else ""
            last = _FROM.sub("", last).strip().strip(".")
            rows.append(dict(arm=arm, dataset=dataset, cond=cond, uid=e.uid, kind=e.kind, k=e.k,
                             pred=pred, gold=e.answer, em=M.em(pred, golds), f1=M.f1(pred, golds),
                             cite_ok=float(np.mean([t in e.support for t in cites])) if cites else float("nan"),
                             last_hop_match=float(M._norm_ans(last) == M._norm_ans(pred)) if lines else float("nan"),
                             chain=c))
        _log(f"  [{arm}] {dataset}/{cond}: n={len(items)} EM={100 * np.mean([r['em'] for r in rows if r['cond'] == cond and r['arm'] == arm]):.1f}")
    return rows


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (arm, ds, cond), g in df.groupby(["arm", "dataset", "cond"]):
        m, lo, hi = M.boot_ci(g.em.values)
        out.append(dict(arm=arm, dataset=ds, cond=cond, n=len(g), em=100 * m, lo=100 * lo, hi=100 * hi,
                        f1=100 * g.f1.mean(), cite_ok=g.cite_ok.mean(), last_hop_match=g.last_hop_match.mean()))
    return pd.DataFrame(out)


# ----------------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=["musique", "wiki2"], required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--arms", nargs="+", default=["p0", "s_ans", "s_plain", "s_struct"])
    ap.add_argument("--n_train", type=int, default=2000)
    ap.add_argument("--n_eval", type=int, default=500)
    ap.add_argument("--max_prompt_tokens", type=int, default=1500)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--mb", type=int, default=1)
    ap.add_argument("--accum", type=int, default=16)
    ap.add_argument("--bs", type=int, default=6)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--conds", nargs="+", default=["orig"])
    ap.add_argument("--train_types", nargs="+", default=["compositional", "inference"])
    ap.add_argument("--eval_types", nargs="+", default=["compositional", "inference"])
    ap.add_argument("--device_map", default=None)
    ap.add_argument("--max_minutes", type=float, default=600.0)
    args = ap.parse_args()
    os.makedirs(args.outdir, exist_ok=True)
    json.dump(vars(args), open(os.path.join(args.outdir, "config.json"), "w"), indent=1)

    if args.dataset == "musique":
        train = load_musique_qa("train")
        dev = {c: load_musique_qa("dev", n_dist={"orig": 4, "dist8": 8}[c]) for c in args.conds}
        rng = np.random.default_rng(3)   # same draw as the round-9 MuSiQue runs (seed 0 + 3)
        pick = rng.choice(len(dev["orig"] if "orig" in dev else next(iter(dev.values()))), args.n_eval,
                          replace=False)
        evals = {c: [v[i] for i in pick] for c, v in dev.items()}
    else:
        train = load_wiki2_qa("train", tuple(args.train_types))
        val = load_wiki2_qa("validation", tuple(args.eval_types))
        pick = np.random.default_rng(3).choice(len(val), min(args.n_eval, len(val)), replace=False)
        evals = {"orig": [val[i] for i in pick]}
    _log(f"{args.dataset}: {len(train)} train, eval {[len(v) for v in evals.values()]}")

    rng = random.Random(args.seed)
    rng.shuffle(train)
    all_rows: List[Dict] = []
    info: Dict[str, Dict] = {}
    for arm in args.arms:
        if (time.time() - T0) / 60 > args.max_minutes:
            _log(f"[{arm}] SKIPPED: past max_minutes")
            continue
        _log(f"=== arm {arm}")
        lm = LM(args.model, device_map=args.device_map)
        if arm != "p0":
            system = {"s_ans": SYS_ANS, "s_plain": SYS_PLAIN, "s_struct": SYS_STRUCT}[arm]
            data, dropped = [], 0
            for e in train:
                t = target(e, arm)
                if t is None:
                    dropped += 1
                    continue
                p = lm.prompt(msgs_zero(system, e))
                if len(lm.tok(p, add_special_tokens=False).input_ids) > args.max_prompt_tokens:
                    dropped += 1
                    continue
                data.append((p, t))
                if len(data) >= args.n_train:
                    break
            _log(f"  {len(data)} training examples ({dropped} skipped: no citation / too long)")
            info[arm] = train_sft(lm, data, args)
            info[arm]["skipped"] = dropped
        all_rows += evaluate(lm, arm, args.dataset, evals, args)
        df = pd.DataFrame(all_rows)
        df.to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
        summarize(df).round(3).to_csv(os.path.join(args.outdir, "summary.csv"), index=False)
        json.dump(info, open(os.path.join(args.outdir, "summary.json"), "w"), indent=1)
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    if all_rows:
        print(summarize(pd.DataFrame(all_rows)).round(2).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile hgcd/cfrl.py
"""Round 11: causal training methods on CLUTRR, on top of supervised fine-tuning (hgcd/cftrain.py).

Arms (each trains its own SFT stage first; the stage-1 model is evaluated too):
  s_plain   SFT on gold chains (control, as in round 10)
  s_ans     SFT on the answer only (does the chain matter at all after fine-tuning?)
  s_dpo     SFT, then DPO with programmatic counterfactual chains (FRODO-style inference module):
            rejected = gold hops followed by a hop that states a WRONG answer / a corrupted hop / a wrong answer
  s_grpo    SFT, then GRPO with a verifiable reward = answer correctness + 0.3 * hop fidelity (process reward
            from the gold reasoning graph), token-level loss and dynamic sampling (DAPO ingredients)
Extra causal measurements on the chain-writing models (n=120 questions of the unseen k=2-4 task):
  wrong-copy follow rate (round-3 protocol T_copy-wrong) and the do_total effect of non-final hops.

    python -m hgcd.cfrl --model Qwen/Qwen2.5-3B-Instruct --outdir out --arms s_ans s_dpo
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import os
import random
import re
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from . import chain as C
from . import data as D
from .causaldec import _first_rel, induce_table, parse_triples
from .cftrain import (LORA_TARGETS, SETS, _tokenize, chain_text, evaluate, extend_table, msgs_zero,
                      summarize, train_sft)
from .experiments import _dist
from .lm import LM
from .prompts import CLUTRR_SYSTEM, clutrr_messages

T0 = time.time()
ANS_SYSTEM = ("You solve family-relation puzzles. Reply with the final answer only, in the form "
              "'Answer: X is Y's <relation>'.")


def _log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ----------------------------------------------------------------------------- data

def ans_text(e: D.ClutrrExample) -> str:
    return f"Answer: {e.target} is {e.source}'s {e.answer}."


def _hops(e: D.ClutrrExample, rel_override: Optional[Dict[int, str]] = None) -> List[str]:
    out = []
    for i, h in enumerate(e.gold_hops):
        rel = (rel_override or {}).get(i, h.rel)
        out.append(f"Hop {i + 1}: {h.head}'s {rel} is {h.tail}. => {h.tail}")
    return out


def cf_pair(e: D.ClutrrExample, rels: List[str], rng: random.Random) -> Tuple[str, str]:
    """(chosen, rejected): chosen is the gold chain; rejected is a counterfactual chain."""
    chosen = chain_text(e, {}, False)
    wrong = rng.choice([r for r in rels if r != e.answer])
    kind = rng.random()
    if kind < 0.4:      # copies a wrong stated answer (mirrors the T_copy-wrong test)
        stated = f"Hop {e.k + 1}: So {e.target} is {e.source}'s relative: => {wrong}"
        rej = "\n".join(_hops(e) + [stated, f"Answer: {e.target} is {e.source}'s {wrong}."])
    elif kind < 0.7:    # one corrupted hop and an answer that follows it
        j = rng.randrange(e.k)
        alt = rng.choice([r for r in rels if r != e.gold_hops[j].rel])
        rej = "\n".join(_hops(e, {j: alt}) + [f"Answer: {e.target} is {e.source}'s {wrong}."])
    else:               # right hops, wrong answer
        rej = "\n".join(_hops(e) + [f"Answer: {e.target} is {e.source}'s {wrong}."])
    return chosen, rej


# ----------------------------------------------------------------------------- DPO

def _seq_logp(pm, ids, lab, att, cuda):
    with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
        logits = pm(input_ids=ids, attention_mask=att).logits
    lp = torch.log_softmax(logits[:, :-1].float(), -1)
    tgt = lab[:, 1:]
    tok_lp = lp.gather(-1, tgt.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    m = (tgt != -100).float()
    return (tok_lp * m).sum(-1), m.sum(-1)


def _pad(batch, pad_id, dev):
    L = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x)
        lab[i, :len(y)] = torch.tensor(y)
        att[i, :len(x)] = 1
    return ids.to(dev), lab.to(dev), att.to(dev)


def train_dpo(lm: LM, pairs: List[Tuple[str, str, str]], args) -> Dict:
    """LoRA DPO on top of the (merged) SFT model; the reference is the SFT model (adapter disabled).
    A small NLL term on the chosen chain (RPO) keeps its likelihood from collapsing."""
    from peft import LoraConfig, get_peft_model
    tok, model = lm.tok, lm.model
    cuda = torch.cuda.is_available()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0,
                                          target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    params = [p for p in pm.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    opt = torch.optim.AdamW(params, lr=args.dpo_lr, weight_decay=0.0)
    enc = [(_tokenize(tok, p, c), _tokenize(tok, p, r)) for p, c, r in pairs]
    n_steps = math.ceil(len(enc) / args.dpo_accum)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / 5) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / n_steps))))
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    pm.train()
    rng = random.Random(args.seed + 1)
    order = list(range(len(enc)))
    rng.shuffle(order)
    stats, step, t_start = [], 0, time.time()
    for n, i in enumerate(order):
        batch = [enc[i][0], enc[i][1]]
        ids, lab, att = _pad(batch, tok.pad_token_id, lm.device)
        with torch.no_grad(), pm.disable_adapter():
            ref, _ = _seq_logp(pm, ids, lab, att, cuda)
        pol, ntok = _seq_logp(pm, ids, lab, att, cuda)
        margin = args.dpo_beta * ((pol[0] - pol[1]) - (ref[0] - ref[1]))
        loss = -torch.nn.functional.logsigmoid(margin) + args.dpo_nll * (-pol[0] / ntok[0])
        scaler.scale(loss / args.dpo_accum).backward()
        stats.append((loss.item(), float(margin.item() > 0)))
        if (n + 1) % args.dpo_accum == 0 or n + 1 == len(order):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            if step % 10 == 0 or step == n_steps:
                s = np.array(stats[-10 * args.dpo_accum:])
                _log(f"  dpo step {step}/{n_steps} loss {s[:, 0].mean():.4f} pref-acc {s[:, 1].mean():.2f} "
                     f"({(time.time() - t_start) / 60:.1f} min)")
    pm.eval()
    merged = pm.merge_and_unload()
    merged.gradient_checkpointing_disable()
    merged.config.use_cache = True
    merged.eval()
    lm.model = merged
    return dict(steps=step, n_pairs=len(enc), minutes=(time.time() - t_start) / 60)


# ----------------------------------------------------------------------------- GRPO

def reward_fn(e: D.ClutrrExample, text: str, vocab: List[str]) -> Tuple[float, float]:
    m = re.search(r"Answer:\s*(.*)", text)
    ans = _first_rel(m.group(1), vocab) if m else None
    gold_t = {(C.norm(h.head), h.rel, C.norm(h.tail)) for h in e.gold_hops}
    trip = parse_triples(text)
    fid = float(np.mean([(C.norm(x), r, C.norm(y)) in gold_t for x, r, y in trip])) if trip else 0.0
    correct = float(ans == e.answer)
    return correct + 0.3 * fid - (0.0 if m else 0.2), correct


def hard_pool(lm: LM, sample: List[D.ClutrrExample], vocab: List[str], args) -> List[D.ClutrrExample]:
    """Keep prompts the SFT model does not always solve (pass rate < 1 over 4 samples): where RL has signal."""
    rng = random.Random(args.seed + 9)
    cand = rng.sample(sample, min(args.rl_hard_n, len(sample)))
    prompts = [lm.prompt(msgs_zero(CLUTRR_SYSTEM, e)) for e in cand]
    outs = []
    for k in sorted(set(e.k for e in cand)):
        idx = [i for i, e in enumerate(cand) if e.k == k]
        o = lm.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 30, do_sample=True, n=4,
                        temperature=1.0, batch_size=max(1, args.bs // 2), stop="<|im_end|>")
        for t, i in enumerate(idx):
            outs.append((i, [reward_fn(cand[i], x, vocab)[1] for x in o[t * 4:(t + 1) * 4]]))
    hard = [cand[i] for i, r in outs if np.mean(r) < 1.0]
    _log(f"  hard pool: {len(hard)}/{len(cand)} prompts with pass rate < 1 "
         f"(mean pass {np.mean([np.mean(r) for _, r in outs]):.2f})")
    return hard if len(hard) >= 16 else cand


def train_grpo(lm: LM, sample: List[D.ClutrrExample], vocab: List[str], args) -> Dict:
    """On-policy group-relative policy gradient (one update per rollout batch, so the importance ratio is 1)
    with token-level loss, no KL term and dynamic sampling of groups with non-zero reward variance."""
    from peft import LoraConfig, get_peft_model
    tok, model = lm.tok, lm.model
    cuda = torch.cuda.is_available()
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0,
                                          target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    params = [p for p in pm.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    opt = torch.optim.AdamW(params, lr=args.rl_lr, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    rng = random.Random(args.seed + 2)
    if args.rl_hard:
        sample = hard_pool(lm, sample, vocab, args)
    by_k = {k: [e for e in sample if e.k == k] or [e for e in sample] for k in (2, 3, 4)}
    gen_eos = lm.model.generation_config.eos_token_id
    stop_ids = {tok.eos_token_id, tok.pad_token_id} | set(gen_eos if isinstance(gen_eos, (list, tuple)) else [gen_eos])
    stop_ids.discard(None)
    log, t_start = [], time.time()
    P, G = args.rl_p, args.rl_g
    for step in range(1, args.rl_steps + 1):
        if (time.time() - t_start) / 60 > args.rl_minutes:
            _log(f"  grpo: time budget reached at step {step - 1}")
            break
        exs = [rng.choice(by_k[rng.choices([2, 3, 4], [0.2, 0.3, 0.5])[0]]) for _ in range(P)]
        prompts = [lm.prompt(msgs_zero(CLUTRR_SYSTEM, e)) for e in exs]
        enc = lm._enc(prompts)
        pm.eval()
        model.config.use_cache = True
        with torch.no_grad():
            g = pm.generate(**enc, max_new_tokens=26 * (max(e.k for e in exs) + 2) + 30, do_sample=True,
                            temperature=1.0, top_p=1.0, top_k=0, num_return_sequences=G,
                            pad_token_id=tok.pad_token_id)
        Lp = enc.input_ids.shape[1]
        comp = g[:, Lp:]
        texts = tok.batch_decode(comp, skip_special_tokens=True)
        R = np.zeros(P * G)
        ok = np.zeros(P * G)
        for i, t in enumerate(texts):
            R[i], ok[i] = reward_fn(exs[i // G], t, vocab)
        A = np.zeros(P * G)
        keep = []
        for p in range(P):
            r = R[p * G:(p + 1) * G]
            if r.std() > 1e-6:
                A[p * G:(p + 1) * G] = (r - r.mean()) / (r.std() + 1e-4)
                keep += list(range(p * G, (p + 1) * G))
        log.append((R.mean(), ok.mean(), len(keep) // G))
        if keep:
            mask = torch.zeros_like(comp, dtype=torch.float32)
            for i in range(comp.shape[0]):
                pos = [j for j, t in enumerate(comp[i].tolist()) if t in stop_ids]
                mask[i, :(pos[0] + 1 if pos else comp.shape[1])] = 1.0   # the stop token is a generated token
            attn = torch.cat([enc.attention_mask.repeat_interleave(G, 0),
                              (mask > 0).long().to(enc.attention_mask.device)], 1)
            pm.train()
            model.config.use_cache = False
            n_tok = mask[keep].sum().clamp(min=1.0)
            for s0 in range(0, len(keep), args.rl_mb):
                idx = keep[s0:s0 + args.rl_mb]
                ids = g[idx].to(lm.device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
                    logits = pm(input_ids=ids, attention_mask=attn[idx].to(lm.device)).logits
                lp = torch.log_softmax(logits[:, Lp - 1:-1].float(), -1)
                tl = lp.gather(-1, comp[idx].to(lm.device).unsqueeze(-1)).squeeze(-1)
                adv = torch.tensor(A[idx], dtype=torch.float32, device=lm.device).unsqueeze(1)
                loss = -(adv * tl * mask[idx].to(lm.device)).sum() / n_tok.to(lm.device)
                scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
        if step % 10 == 0:
            a = np.array(log[-10:])
            _log(f"  grpo step {step}/{args.rl_steps} reward {a[:, 0].mean():.3f} acc {a[:, 1].mean():.3f} "
                 f"kept {a[:, 2].mean():.1f}/{P} ({(time.time() - t_start) / 60:.1f} min)")
    pm.eval()
    merged = pm.merge_and_unload()
    merged.gradient_checkpointing_disable()
    merged.config.use_cache = True
    merged.eval()
    lm.model = merged
    a = np.array(log) if log else np.zeros((1, 3))
    return dict(steps=len(log), reward_first10=float(a[:10, 0].mean()), reward_last10=float(a[-10:, 0].mean()),
                acc_first10=float(a[:10, 1].mean()), acc_last10=float(a[-10:, 1].mean()),
                minutes=(time.time() - t_start) / 60)


# ----------------------------------------------------------------------------- causal measurements

def causal_extras(lm: LM, arm: str, rels: List[str], n: int, seed: int = 7) -> List[Dict]:
    """T_copy-wrong follow rate and mean do_total effect of non-final gold hops (chain-writing models)."""
    task, split, ks, npk, sd = SETS["c6"]
    pool = D.load_clutrr(split, task=task)
    exs = D.sample_by_k(pool, ks, npk, sd)
    random.Random(0).shuffle(exs)
    exs = exs[:n]
    cands = [" " + r for r in rels]
    rng = random.Random(f"wc-{seed}")
    follow, ps, follow_ps, eff = [], [], [], []
    for e in exs:
        base = lm.prompt(msgs_zero(CLUTRR_SYSTEM, e))
        pre = e.answer_prefix()
        hops = [C.Hop(i + 1, h.text(), h.tail) for i, h in enumerate(e.gold_hops)]
        g = rels.index(e.answer)
        p_full = _dist(lm, base + C.render_chain(hops) + pre, cands)
        ps.append(float(np.argmax(p_full) == g))
        wrong = rng.choice([r for r in rels if r != e.answer])
        w_hops = hops + [C.Hop(len(hops) + 1, f"So {e.target} is {e.source}'s relative:", wrong)]
        p_w = _dist(lm, base + C.render_chain(w_hops) + pre, cands)
        f = float(np.argmax(p_w) == rels.index(wrong))
        follow.append(f)
        if ps[-1]:
            follow_ps.append(f)
        for j in range(len(hops) - 1):
            p_m = _dist(lm, base + C.render_chain(C.do_total(hops, j)) + pre, cands)
            eff.append(p_full[g] - p_m[g])
    mk = lambda metric, v: dict(arm=arm, metric=metric, value=float(np.mean(v)) if len(v) else float("nan"),
                                n=len(v))
    out = [mk("wrong_copy_follow_all", follow), mk("wrong_copy_follow_given_ps1", follow_ps),
           mk("ps_plain_chain", ps), mk("do_total_effect_nonfinal_hops", eff)]
    _log(f"  [{arm}] wrong-copy follow {100 * out[0]['value']:.1f}% (PS=1: {100 * out[1]['value']:.1f}%, "
         f"n={out[1]['n']}); do_total effect {100 * out[3]['value']:.1f} pts")
    return out


def sanity(lm: LM):
    enc = lm._enc([lm.prompt([{"role": "user", "content": "Say hello."}])])
    with torch.no_grad():
        lg = lm.model(**enc).logits
    if not torch.isfinite(lg).all():
        raise RuntimeError("non-finite logits in fp16: this model cannot be run in fp16 on this GPU")


def eval_think(lm: LM, arm: str, vocab: List[str], args, table) -> List[Dict]:
    """Native thinking mode (Qwen3): long generation, answer read after </think>; a random subset of each set."""
    rows = []
    for name in ("c6", "c5"):
        task, split, ks, npk, seed = SETS[name]
        pool = D.load_clutrr(split, task=task)
        base = D.sample_by_k(pool, ks, npk, seed)
        random.Random(1).shuffle(base)
        ex = base[:args.think_n]
        prompts = [lm.prompt(msgs_zero(ANS_SYSTEM, e), thinking=True) for e in ex]
        outs = lm.generate(prompts, max_new_tokens=args.think_tokens, batch_size=args.bs, stop="<|im_end|>")
        fin = 0
        for e, o in zip(ex, outs):
            done = "</think>" in o
            fin += done
            tail = o.split("</think>")[-1] if done else ""
            m = re.search(r"Answer:\s*(.*)", tail)
            pred = _first_rel(m.group(1) if m else tail, vocab) if done else None
            rows.append(dict(arm=arm, set=name, cond="orig", uid=e.uid, k=e.k, pred=pred, gold=e.answer,
                             correct=float(pred == e.answer), hop_fidelity=float("nan"), step_acc=float("nan"),
                             last_step_ok=float("nan"), covered=int(covered(e, table)), chain=o[-600:],
                             finished=float(done)))
        acc = np.mean([r["correct"] for r in rows if r["set"] == name])
        _log(f"  [{arm}] {name}: n={len(ex)} finished={fin / len(ex):.2f} acc={100 * acc:.1f}")
    return rows


def star_data(args, sample: List[D.ClutrrExample], vocab: List[str]) -> Tuple[List[Tuple[str, str]], Dict]:
    """Rejection-sampled chains from a teacher (no gold chains used): keep the first sample whose answer is right."""
    teacher = LM(args.teacher, device_map=args.teacher_device_map)
    exs = sample[:args.star_n]
    prompts = [teacher.prompt(msgs_zero(CLUTRR_SYSTEM, e)) for e in exs]
    outs = []
    for k in sorted(set(e.k for e in exs)):
        idx = [i for i, e in enumerate(exs) if e.k == k]
        o = teacher.generate([prompts[i] for i in idx], max_new_tokens=26 * (k + 2) + 30, do_sample=True,
                             n=args.star_k, temperature=0.8, batch_size=max(1, args.bs // 2), stop="<|im_end|>")
        for t, i in enumerate(idx):
            outs.append((i, o[t * args.star_k:(t + 1) * args.star_k]))
        _log(f"  teacher k={k}: {len(idx)} questions")
    data, ok_any, ok_all = [], 0, []
    for i, cands in outs:
        res = [reward_fn(exs[i], c, vocab)[1] for c in cands]
        ok_all += res
        good = [c for c, r in zip(cands, res) if r == 1.0 and "Answer:" in c and "Hop" in c]
        if good:
            ok_any += 1
            data.append((i, good[0].strip()))
    info = dict(teacher_sample_acc=float(np.mean(ok_all)), questions=len(exs), kept=len(data),
                kept_frac=len(data) / len(exs))
    del teacher
    gc.collect()
    torch.cuda.empty_cache()
    return data, info, exs


# ----------------------------------------------------------------------------- main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--arms", nargs="+", default=["s_ans", "s_dpo"])
    ap.add_argument("--sets", nargs="+", default=["c4", "c6", "c5"])
    ap.add_argument("--n_train", type=int, default=3000)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--mb", type=int, default=2)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--eval_scale", type=float, default=1.0)
    ap.add_argument("--robust", type=int, default=1)
    ap.add_argument("--extras_n", type=int, default=120)
    ap.add_argument("--device_map", default=None)
    ap.add_argument("--max_minutes", type=float, default=600.0)
    # DPO
    ap.add_argument("--dpo_pairs", type=int, default=1500)
    ap.add_argument("--dpo_beta", type=float, default=0.1)
    ap.add_argument("--dpo_lr", type=float, default=2e-5)
    ap.add_argument("--dpo_accum", type=int, default=8)
    ap.add_argument("--dpo_nll", type=float, default=0.2)
    # GRPO
    ap.add_argument("--rl_steps", type=int, default=200)
    ap.add_argument("--rl_p", type=int, default=8)
    ap.add_argument("--rl_g", type=int, default=8)
    ap.add_argument("--rl_mb", type=int, default=8)
    ap.add_argument("--rl_lr", type=float, default=2e-5)
    ap.add_argument("--rl_minutes", type=float, default=150.0)
    ap.add_argument("--rl_hard", type=int, default=0)
    ap.add_argument("--rl_hard_n", type=int, default=500)
    # Qwen3 / distillation
    ap.add_argument("--teacher", default=None)
    ap.add_argument("--teacher_device_map", default="auto")
    ap.add_argument("--star_n", type=int, default=1500)
    ap.add_argument("--star_k", type=int, default=2)
    ap.add_argument("--think_n", type=int, default=100)
    ap.add_argument("--think_tokens", type=int, default=1800)
    args = ap.parse_args()
    os.makedirs(args.outdir, exist_ok=True)
    json.dump(vars(args), open(os.path.join(args.outdir, "config.json"), "w"), indent=1)

    train = D.load_clutrr("train")
    vocab_l = D.clutrr_relations(train)
    table = extend_table(train, induce_table(train), vocab_l)
    pool = [e for e in train if 2 <= e.k <= 4]
    sample = D.sample_by_k(pool, [2, 3, 4], args.n_train // 3, args.seed)
    _log(f"{len(sample)} train chains (k=2-4)")

    all_rows: List[Dict] = []
    extras: List[Dict] = []
    info: Dict[str, Dict] = {}
    plain_mx = lambda k: 26 * (k + 2) + 16
    zero = lambda e: msgs_zero(CLUTRR_SYSTEM, e)

    def flush():
        df = pd.DataFrame(all_rows)
        df.to_csv(os.path.join(args.outdir, "decisions.csv"), index=False)
        summarize(df).round(3).to_csv(os.path.join(args.outdir, "summary.csv"), index=False)
        pd.DataFrame(extras).to_csv(os.path.join(args.outdir, "extras.csv"), index=False)
        json.dump(info, open(os.path.join(args.outdir, "summary.json"), "w"), indent=1)

    for arm in args.arms:
        if (time.time() - T0) / 60 > args.max_minutes:
            _log(f"[{arm}] SKIPPED: past max_minutes")
            continue
        _log(f"=== arm {arm}")
        star_info = None
        if arm == "s_star":
            star, star_info, star_exs = star_data(args, sample, vocab_l)
        lm = LM(args.model, device_map=args.device_map)
        sanity(lm)
        if arm == "p_fs":
            all_rows += evaluate(lm, arm, lambda e: clutrr_messages(e.story_text, e.source, e.target), args,
                                 table, vocab_l, args.sets, bool(args.robust), plain_mx)
            info[arm] = {}
        elif arm == "p_think":
            all_rows += eval_think(lm, arm, vocab_l, args, table)
            info[arm] = {}
        elif arm == "s_star":
            data = [(lm.prompt(zero(star_exs[i])), t) for i, t in star]
            info[arm] = train_sft(lm, data, args)
            info[arm]["star"] = star_info
            all_rows += evaluate(lm, arm, zero, args, table, vocab_l, args.sets, bool(args.robust), plain_mx)
        elif arm == "s_ans":
            data = [(lm.prompt(msgs_zero(ANS_SYSTEM, e)), ans_text(e)) for e in sample]
            info[arm] = train_sft(lm, data, args)
            all_rows += evaluate(lm, arm, lambda e: msgs_zero(ANS_SYSTEM, e), args, table, vocab_l, args.sets,
                                 bool(args.robust), lambda k: 0)
        else:
            data = [(lm.prompt(zero(e)), chain_text(e, table, False)) for e in sample]
            info[arm] = train_sft(lm, data, args)
            stage1 = "s_plain" if arm == "s_plain" else "sft_before_" + arm.split("_")[1]
            all_rows += evaluate(lm, stage1, zero, args, table, vocab_l, args.sets, bool(args.robust), plain_mx)
            extras += causal_extras(lm, stage1, vocab_l, args.extras_n)
            flush()
            if arm == "s_dpo":
                rng = random.Random(args.seed + 5)
                pairs = []
                for e in rng.sample(sample, min(args.dpo_pairs, len(sample))):
                    c, r = cf_pair(e, vocab_l, rng)
                    pairs.append((lm.prompt(zero(e)), c, r))
                info[arm]["dpo"] = train_dpo(lm, pairs, args)
            elif arm == "s_grpo":
                info[arm]["grpo"] = train_grpo(lm, sample, vocab_l, args)
            if arm != "s_plain":
                all_rows += evaluate(lm, arm, zero, args, table, vocab_l, args.sets, bool(args.robust), plain_mx)
                extras += causal_extras(lm, arm, vocab_l, args.extras_n)
        flush()
        del lm
        gc.collect()
        torch.cuda.empty_cache()
    if all_rows:
        print(summarize(pd.DataFrame(all_rows)).round(3).to_string())
        print(pd.DataFrame(extras).round(3).to_string())


if __name__ == "__main__":
    main()


In [ ]:
import os
# ---- configuration -------------------------------------------------------------
NOTEBOOK = "R12_q3_8b"
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
MAX_START_HOURS = 8.5   # never START a job after this many hours (Kaggle kills sessions at 12 h
                        # and then saves nothing); finished jobs are always kept
# PHASES run one after another; inside a phase, each GPU queue runs in parallel.
# job = (label, model, preset, extra CLI args)
PHASES = {
    "q8": {"0,1": [("qwen3-8b", "Qwen/Qwen3-8B", "cfrl",
                    ("--arms", "p_fs", "p_think", "s_plain", "s_ans", "--device_map", "auto", "--mb", "1", "--accum", "16",
                     "--bs", "8", "--think_n", "60", "--think_tokens", "1500", "--seed", "0", "--max_minutes", "560"))]},
}


## Run (both GPUs)

In [ ]:
import subprocess, sys, threading, time, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
ngpu = torch.cuda.device_count()

def run_job(label, model, preset, extra, gpus):
    if (time.time() - T0) / 3600 > MAX_START_HOURS:
        print(f"[{label}] SKIPPED: past MAX_START_HOURS"); return
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus)
    out = f"{OUT}/{preset}_{label}"
    with open(out + ".log", "w") as log:
        cmd = ([sys.executable, "-m", "hgcd.cfrl", "--model", model, "--outdir", out, *extra]
               if preset.startswith("cfrl") else
               [sys.executable, "-m", "hgcd.egen", "--model", model, "--outdir", out, *extra]
               if preset.startswith("egen") else
               [sys.executable, "-m", "hgcd.cftrain", "--model", model, "--outdir", out, *extra]
               if preset.startswith("cftrain") else
               [sys.executable, "-m", "hgcd.causaldec", "--model", model, "--outdir", out, *extra]
               if preset.startswith("causaldec") else
               [sys.executable, "-m", "hgcd.pipeline", "--preset", preset, "--model", model,
                "--outdir", out, *extra])
        rc = subprocess.call(cmd,
                             env=env, stdout=log, stderr=subprocess.STDOUT)
    print(f"[{label}] exit {rc} after {(time.time() - T0) / 3600:.1f} h; log tail:")
    print("".join(open(out + ".log").readlines()[-14:]))

def run_queue(gpus, queue):
    for job in queue:
        run_job(*job, gpus)

for name, queues in PHASES.items():
    print("=" * 20, "phase", name)
    if ngpu < 2 and len(queues) > 1:  # single GPU: serialise
        queues = {"0": [j for q in queues.values() for j in q]}
    ths = [threading.Thread(target=run_queue, args=(g, q)) for g, q in queues.items()]
    for t in ths: t.start()
    for t in ths: t.join()


## Results

In [ ]:
import json, glob, pandas as pd
from IPython.display import display
from hgcd import report as R
dirs = {os.path.basename(d): d for d in sorted(glob.glob(f"{OUT}/*")) if os.path.isdir(d)}
rel = {k: v for k, v in dirs.items() if k.startswith("reliance")}
if rel:
    df = R.reliance_table(rel, f"{OUT}/tab_reliance.tex", f"{OUT}/reliance_vs_scale.pdf")
    display(df)
    print(open(f"{OUT}/tab_reliance.tex").read())
for k, d in dirs.items():
    if os.path.exists(f"{d}/summary.csv"):
        print("=" * 10, k); print(pd.read_csv(f"{d}/summary.csv").round(3).to_string())
    if os.path.exists(f"{d}/summary.json"):
        s = json.load(open(f"{d}/summary.json"))
        print(k, "copy_wrong:", s.get("copy_wrong"))
    for f in sorted(glob.glob(f"{d}/tab_*.tex")):
        print("=" * 10, k, os.path.basename(f)); print(open(f).read())


In [ ]:
import shutil
shutil.make_archive(f'{OUT}_{NOTEBOOK}', 'zip', OUT)
print('download:', f'{OUT}_{NOTEBOOK}.zip')